# 🧬 Banco de Pruebas en Google Colab — Adaptive MorphPooling RGB vs. MaxPool2d
**Trabajo Final de Grado / Tesis de Ingeniería** · *Christian Yegros & Prof. Dr. José Luis Vázquez Noguera (FP-UNA)*

Este notebook está **100% autocontenido** (no necesitas subir los archivos `.py` por separado) y se conecta directamente a tu carpeta de datasets en Google Drive (`1AIX0huqYBfBH3v8Sc47xeFcGxYYbslQH`).
Incluye la versión actualizada del código con:
- **Partición de Dominio del Paper (`W0`, `W1`, `W4`, `W9`, `W16`, `W25`, `W36`, `W49`, `W64`)** con unión exacta de sub-regiones por contacto.
- **Los 8 Descriptores Estadísticos ($\phi$)** incluyendo **Entropía de Shannon (`ENT`, 0–255)**, `SMO`, `MEAN`, `VAR`, `MO1`, `MO2`, `MIN` y `MAX`.
- **Guardado automático en Google Drive** de todos los reportes `.html` y `.json` para no perder ningún resultado aunque Colab se desconecte.

---

### 🚀 Instrucciones Rápidas (3 pasos):
1. **Activa la GPU en Colab:** Ve arriba al menú **`Entorno de ejecución` $\rightarrow$ `Cambiar tipo de entorno de ejecución`** y selecciona **`GPU T4`**.
2. **Usa la misma cuenta de Google de tu carpeta (`/u/4/`):** Abre Colab con esa misma cuenta de Google (o comparte esa carpeta con tu cuenta de Colab y pulsa *"Organizar $\rightarrow$ Añadir acceso directo a Mi unidad"*).
3. **Dale Play ▶ en orden a las Celdas 1, 2 y 3:** En la **Celda 3** tienes listas desplegables idénticas a la interfaz gráfica de Windows.

In [ ]:
#@title 🔌 1. Verificar GPU y Conectar Carpeta de Datasets en Google Drive { display-mode: "form" }
import os
import torch
from google.colab import drive

DRIVE_FOLDER_ID = "1AIX0huqYBfBH3v8Sc47xeFcGxYYbslQH" #@param {type:"string"}
ruta_manual_opcional = "" #@param {type:"string"}

print("=" * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detectada: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ ATENCIÓN: No se detectó GPU. Ve a 'Entorno de ejecución -> Cambiar tipo de entorno de ejecución' y elige GPU T4.")
print("=" * 65)

# Montar Google Drive
drive.mount("/content/drive", force_remount=False)

def find_drive_folder_by_id(target_id, manual_hint=""):
    if manual_hint.strip():
        cand = manual_hint.strip()
        if not cand.startswith("/content/drive"):
            cand = os.path.join("/content/drive/MyDrive", cand.lstrip("/"))
        if os.path.isdir(cand):
            return cand

    # En Google Colab, Drive FUSE expone el ID real de cada carpeta en el atributo 'user.drive.id'
    search_roots = ["/content/drive/MyDrive", "/content/drive/Shareddrives"]
    for root_base in search_roots:
        if not os.path.isdir(root_base):
            continue
        for current_root, dirs, _ in os.walk(root_base, followlinks=True):
            rel_depth = current_root[len(root_base):].count(os.sep)
            try:
                xid = os.getxattr(current_root, "user.drive.id").decode("utf-8", errors="ignore").strip()
                if xid == target_id:
                    return current_root
            except OSError:
                pass
            if rel_depth >= 4:
                dirs[:] = []
    return None

DATASETS_DRIVE_DIR = find_drive_folder_by_id(DRIVE_FOLDER_ID, ruta_manual_opcional)

if DATASETS_DRIVE_DIR and os.path.isdir(DATASETS_DRIVE_DIR):
    print(f"\n✅ Carpeta de Google Drive conectada automáticamente:\n   📂 {DATASETS_DRIVE_DIR}\n")
    items = sorted(os.listdir(DATASETS_DRIVE_DIR))
    print(f"📦 Elementos encontrados en tu carpeta ({len(items)}):")
    for it in items:
        full_p = os.path.join(DATASETS_DRIVE_DIR, it)
        if os.path.isdir(full_p):
            print(f"   📁 [Carpeta]  {it}")
        else:
            sz_mb = os.path.getsize(full_p) / (1024 * 1024)
            print(f"   🗜️ [Archivo]  {it} ({sz_mb:.1f} MB)")
else:
    print("\n⚠️ No se encontró la carpeta en 'Mi unidad' de la cuenta actual de Colab.")
    print("   Solución rápida:")
    print(f"   1. Abre el enlace: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    print("   2. Inicia sesión en Colab con esa misma cuenta, O haz clic derecho en la carpeta -> 'Organizar' -> 'Añadir acceso directo' a 'Mi unidad'.")
    print("   3. Vuelve a darle Play a esta celda.")

In [ ]:
#@title ⚙️ 2. Instalar Módulos del Proyecto (MorphPool + W0..W64 + Entropía ENT) { display-mode: "form" }
import os, json

WORK_DIR = "/content/proyecto_tesis"
os.makedirs(WORK_DIR, exist_ok=True)

MODULES = json.loads(r'''{"morph_pool_layer.py":"import math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nVALID_PARTITIONS = (\"W0\", \"W1\", \"W4\", \"W9\", \"W16\", \"W25\", \"W36\", \"W49\", \"W64\")\n\n\ndef normalize_partition(partition=\"W0\", stat_window=None):\n    \"\"\"\n    Normaliza el identificador de partición de dominio según Vázquez Noguera et al. (2019):\n      - \u0027W0\u0027  : Estrategia 1 (Vecindad local definida por la ventana B / stat_window).\n      - \u0027W1\u0027  : Estrategia 2 caso especial (1 sub-región = imagen completa / global).\n      - \u0027WX\u0027  : Estrategia 2 (X = M\u0027 * N\u0027 sub-regiones con unión de sub-regiones en contacto con B,\n                ej. W4=2x2, W9=3x3, W16=4x4, W25=5x5, W49=7x7).\n      - \u0027MxN\u0027 : Cuadrícula explícita M\u0027 x N\u0027 sub-regiones (ej. \u00273x3\u0027, \u00275x5\u0027).\n    \"\"\"\n    if str(stat_window).strip().lower() == \"global\" and (\n        partition is None or str(partition).strip().upper() in (\"\", \"W0\")\n    ):\n        return \"W1\"\n\n    if partition is None:\n        return \"W0\"\n\n    raw = str(partition).strip().upper()\n    if raw in (\"\", \"0\", \"W0\", \"NEIGHBORHOOD\", \"VECINDAD\", \"LOCAL\"):\n        return \"W0\"\n    if raw in (\"1\", \"W1\", \"GLOBAL\", \"1X1\"):\n        return \"W1\"\n\n    if raw.isdigit():\n        raw = f\"W{int(raw)}\"\n\n    if raw.startswith(\"W\") and raw[1:].isdigit():\n        n_reg = int(raw[1:])\n        if n_reg == 0:\n            return \"W0\"\n        if n_reg == 1:\n            return \"W1\"\n        root = math.isqrt(n_reg)\n        if root * root != n_reg or root \u003c 1:\n            raise ValueError(\n                f\"Partición \u0027{partition}\u0027 inválida: el número de sub-regiones X en WX debe ser un cuadrado perfecto \"\n                f\"(ej. W0, W1, W4, W9, W16, W25, W49) o formato MxN.\"\n            )\n        return f\"W{n_reg}\"\n\n    if \"X\" in raw:\n        parts = raw.split(\"X\")\n        if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():\n            m_grid, n_grid = int(parts[0]), int(parts[1])\n            if m_grid \u003c 1 or n_grid \u003c 1:\n                raise ValueError(f\"Dimensiones de cuadrícula inválidas en \u0027{partition}\u0027.\")\n            if m_grid == 1 and n_grid == 1:\n                return \"W1\"\n            if m_grid == n_grid:\n                return f\"W{m_grid * n_grid}\"\n            return f\"{m_grid}x{n_grid}\"\n\n    raise ValueError(\n        f\"Estrategia de partición desconocida: \u0027{partition}\u0027. \"\n        f\"Use W0 (vecindad), W1 (global), W4, W9, W16, W25, W49 o MxN.\"\n    )\n\n\ndef parse_partition_grid(partition_norm):\n    \"\"\"Retorna (M\u0027, N\u0027) para una partición normalizada distinta de W0.\"\"\"\n    if partition_norm == \"W1\":\n        return (1, 1)\n    if partition_norm.startswith(\"W\"):\n        n_reg = int(partition_norm[1:])\n        root = math.isqrt(n_reg)\n        return (root, root)\n    m_str, n_str = partition_norm.lower().split(\"x\")\n    return (int(m_str), int(n_str))\n\n\ndef format_domain_label(partition=\"W0\", stat_window=2, kernel_size=2):\n    \"\"\"Genera una etiqueta legible con la nomenclatura del paper (ej. \u0027B2W9 (3x3 sub-regiones)\u0027).\"\"\"\n    p_norm = normalize_partition(partition, stat_window)\n    sw_str = \"global\" if str(stat_window).strip().lower() == \"global\" else str(stat_window or kernel_size)\n\n    if p_norm == \"W0\":\n        b_size = kernel_size if sw_str == \"global\" else int(sw_str)\n        return f\"B{b_size}W0 (Vecindad {b_size}x{b_size})\"\n    if p_norm == \"W1\":\n        b_size = kernel_size if sw_str == \"global\" else int(sw_str)\n        return f\"B{b_size}W1 (Global 1x1)\"\n\n    m_grid, n_grid = parse_partition_grid(p_norm)\n    b_size = kernel_size if sw_str == \"global\" else int(sw_str)\n    n_total = m_grid * n_grid\n    return f\"B{b_size}{p_norm} ({n_total} sub-regiones {m_grid}x{n_grid})\"\n\n\nclass AdaptiveMorphPooling2d(nn.Module):\n    \"\"\"\n    Capa de Pooling Morfológico Adaptativo Vectorial basada en el marco de ordenamiento\n    de Vázquez Noguera et al. (IEEE Access, 2019).\n\n    Soporta las dos estrategias de partición de dominio del artículo (Sección II-A):\n      1) partition=\u0027W0\u0027 (Vecindad / Neighborhood):\n         El dominio D_m corresponde a la ventana local B centrada/ubicada en la posición\n         de filtrado (de tamaño stat_window x stat_window: 2, 3, 5, 7).\n      2) partition=\u0027WX\u0027 (Partición de la imagen en sub-regiones: W1, W4, W9, W16, W25, W49...):\n         La imagen M x N se divide en una matriz de M\u0027 x N\u0027 sub-regiones con floor(M/M\u0027) filas\n         y floor(N/N\u0027) columnas. Para cada ventana B, cualquier sub-región en contacto con al\n         menos un píxel de B forma parte de D_m (unión de sub-regiones en contacto, Figs. 2 y 3).\n         El caso especial W1 corresponde a 1 sola región (imagen completa / global).\n    \"\"\"\n\n    def __init__(self, kernel_size=2, stride=2, metric=\u0027SMO\u0027, stat_window=None, partition=\u0027W0\u0027):\n        super(AdaptiveMorphPooling2d, self).__init__()\n        self.kernel_size = kernel_size     # Ventana de pooling (2x2)\n        self.stride = stride               # Stride de pooling (2)\n        self.metric = metric\n        self.stat_window = stat_window     # None/2, 3, 5, 7 o \u0027global\u0027\n        self.partition = normalize_partition(partition, stat_window)\n        self._layout_cache = {}\n\n    def _get_subregion_layout(self, H, W, device):\n        \"\"\"\n        Precalcula y almacena en caché los rangos de sub-regiones en contacto con cada ventana B\n        según la Estrategia 2 del artículo (Figuras 2 y 3).\n        \"\"\"\n        m_grid, n_grid = parse_partition_grid(self.partition)\n        kh, kw = self.kernel_size, self.kernel_size\n        sh, sw = self.stride, self.stride\n\n        if self.stat_window is None or str(self.stat_window).strip().lower() == \"global\":\n            bw_h, bw_w = kh, kw\n        else:\n            bw_val = int(self.stat_window)\n            bw_h, bw_w = max(kh, bw_val), max(kw, bw_val)\n\n        cache_key = (H, W, kh, kw, sh, sw, bw_h, bw_w, m_grid, n_grid, device)\n        if cache_key in self._layout_cache:\n            return self._layout_cache[cache_key]\n\n        H_out = (H - kh) // sh + 1\n        W_out = (W - kw) // sw + 1\n\n        # Tamaño base floor(M / M\u0027) y floor(N / N\u0027) según Sección II-A.2 del paper\n        h_step = max(1, H // m_grid)\n        w_step = max(1, W // n_grid)\n\n        pad_top = (bw_h - kh) // 2\n        pad_bottom = (bw_h - kh) - pad_top\n        pad_left = (bw_w - kw) // 2\n        pad_right = (bw_w - kw) - pad_left\n\n        # Determinar para cada fila de salida qué sub-regiones verticales entran en contacto con B\n        u_rows = []\n        row_lookup = {}\n        row_map_list = []\n        for i in range(H_out):\n            y0 = max(0, i * sh - pad_top)\n            y1 = min(H - 1, i * sh + kh - 1 + pad_bottom)\n            p0 = min(y0 // h_step, m_grid - 1)\n            p1 = min(y1 // h_step, m_grid - 1)\n            r_start = p0 * h_step\n            r_end = (p1 + 1) * h_step if p1 \u003c m_grid - 1 else H\n            interval = (r_start, r_end)\n            if interval not in row_lookup:\n                row_lookup[interval] = len(u_rows)\n                u_rows.append(interval)\n            row_map_list.append(row_lookup[interval])\n\n        # Determinar para cada columna de salida qué sub-regiones horizontales entran en contacto con B\n        u_cols = []\n        col_lookup = {}\n        col_map_list = []\n        for j in range(W_out):\n            x0 = max(0, j * sw - pad_left)\n            x1 = min(W - 1, j * sw + kw - 1 + pad_right)\n            q0 = min(x0 // w_step, n_grid - 1)\n            q1 = min(x1 // w_step, n_grid - 1)\n            c_start = q0 * w_step\n            c_end = (q1 + 1) * w_step if q1 \u003c n_grid - 1 else W\n            interval = (c_start, c_end)\n            if interval not in col_lookup:\n                col_lookup[interval] = len(u_cols)\n                u_cols.append(interval)\n            col_map_list.append(col_lookup[interval])\n\n        # Agrupar dominios D_m únicos por cantidad de píxeles para vectorizar el cálculo de phi\n        groups_by_area = {}\n        for ur_idx, (r0, r1) in enumerate(u_rows):\n            for uc_idx, (c0, c1) in enumerate(u_cols):\n                area = (r1 - r0) * (c1 - c0)\n                groups_by_area.setdefault(area, []).append((ur_idx, uc_idx, r0, r1, c0, c1))\n\n        compiled_groups = []\n        for area, items in groups_by_area.items():\n            ur_t = torch.tensor([it[0] for it in items], dtype=torch.long, device=device)\n            uc_t = torch.tensor([it[1] for it in items], dtype=torch.long, device=device)\n            slices = [(it[2], it[3], it[4], it[5]) for it in items]\n            compiled_groups.append((area, ur_t, uc_t, slices))\n\n        row_map = torch.tensor(row_map_list, dtype=torch.long, device=device)\n        col_map = torch.tensor(col_map_list, dtype=torch.long, device=device)\n\n        layout = (len(u_rows), len(u_cols), compiled_groups, row_map, col_map)\n        self._layout_cache[cache_key] = layout\n        return layout\n\n    def _compute_phi(self, x_stat):\n        \"\"\"Evalúa la función estadística phi sobre la última dimensión de x_stat.\"\"\"\n        if self.metric == \u0027SMO\u0027:\n            var = torch.var(x_stat, dim=-1, keepdim=True, unbiased=False)\n            return 1.0 - (1.0 / (1.0 + var))\n        elif self.metric == \u0027VAR\u0027:\n            return torch.var(x_stat, dim=-1, keepdim=True, unbiased=False)\n        elif self.metric == \u0027MEAN\u0027:\n            return torch.mean(x_stat, dim=-1, keepdim=True)\n        elif self.metric == \u0027MIN\u0027:\n            return torch.min(x_stat, dim=-1, keepdim=True).values\n        elif self.metric == \u0027MAX\u0027:\n            return torch.max(x_stat, dim=-1, keepdim=True).values\n        elif self.metric == \u0027MO1\u0027:\n            return self._mode_low(x_stat)\n        elif self.metric == \u0027MO2\u0027:\n            return self._mode_high(x_stat)\n        elif self.metric == \u0027ENT\u0027:\n            return self._compute_entropy(x_stat)\n        else:\n            raise ValueError(f\"Descriptor desconocido: {self.metric}\")\n\n    def forward(self, x):\n        # x shape: [B, 3, H, W] -\u003e Se procesa sobre 3 canales RGB\n        B, C, H, W = x.shape\n        kh, kw = self.kernel_size, self.kernel_size\n        sh, sw = self.stride, self.stride\n\n        # --- 1. Extraer ventanas del pooling 2x2 ---\n        x_unfold = F.unfold(x, kernel_size=(kh, kw), stride=(sh, sw))\n        H_out = (H - kh) // sh + 1\n        W_out = (W - kw) // sw + 1\n\n        x_unfold = x_unfold.view(B, C, kh * kw, H_out, W_out)\n        x_win = x_unfold.permute(0, 1, 3, 4, 2)  # [B, C, H_out, W_out, 4]\n\n        # --- 2. Calcular descriptor phi según Estrategia de Partición de Dominio ---\n        p_norm = normalize_partition(self.partition, self.stat_window)\n\n        if p_norm == \"W1\":\n            # Caso global / 1 sola sub-región (imagen completa)\n            x_stat = x.reshape(B, C, 1, 1, H * W)\n            phi = self._compute_phi(x_stat).expand(B, C, H_out, W_out, 1)\n\n        elif p_norm == \"W0\":\n            # Estrategia 1 del paper: Partición basada en vecindad (ventana local)\n            if self.stat_window is None or int(self.stat_window) == self.kernel_size:\n                x_stat = x_win  # [B, C, H_out, W_out, 4]\n            else:\n                sw_int = int(self.stat_window)\n                total_pad = sw_int - kh\n                pad_tl = total_pad // 2\n                pad_br = total_pad - pad_tl\n                x_padded = F.pad(x, (pad_tl, pad_br, pad_tl, pad_br))\n\n                x_stat_unfold = F.unfold(x_padded, kernel_size=(sw_int, sw_int), stride=(sh, sw))\n                x_stat_unfold = x_stat_unfold.view(B, C, sw_int * sw_int, H_out, W_out)\n                x_stat = x_stat_unfold.permute(0, 1, 3, 4, 2)  # [B, C, H_out, W_out, sw_int^2]\n\n            phi = self._compute_phi(x_stat)\n\n        else:\n            # Estrategia 2 del paper: Partición en sub-regiones W_1..W_lambda con unión por contacto\n            n_urows, n_ucols, compiled_groups, row_map, col_map = self._get_subregion_layout(H, W, x.device)\n            phi_grid = torch.empty((B, C, n_urows, n_ucols, 1), dtype=x.dtype, device=x.device)\n\n            for area, ur_t, uc_t, slices in compiled_groups:\n                patches = torch.stack(\n                    [x[:, :, r0:r1, c0:c1].reshape(B, C, area) for (r0, r1, c0, c1) in slices],\n                    dim=2\n                )  # [B, C, n_domains_in_group, area]\n                phi_grp = self._compute_phi(patches)  # [B, C, n_domains_in_group, 1]\n                phi_grid[:, :, ur_t, uc_t, :] = phi_grp\n\n            phi = phi_grid[:, :, row_map[:, None], col_map[None, :], :]  # [B, C, H_out, W_out, 1]\n\n        # --- 3. Vector de pesos psi con manejo de regiones planas ---\n        sum_phi = torch.sum(phi, dim=1, keepdim=True)\n        psi = torch.where(sum_phi \u003e 1e-8, phi / (sum_phi + 1e-8), torch.full_like(phi, 1.0 / C))\n\n        # --- 4. Puntuación escalar T(v) = \u003cv, psi\u003e y selección del píxel ganador intacto ---\n        score = torch.sum(x_win * psi, dim=1)  # [B, H_out, W_out, 4]\n        max_idx = torch.argmax(score, dim=-1, keepdim=True)\n        max_idx_expanded = max_idx.unsqueeze(1).expand(B, C, H_out, W_out, 1)\n\n        out = torch.gather(x_win, dim=-1, index=max_idx_expanded).squeeze(-1)\n        return out  # [B, 3, H_out, W_out]\n\n    # --- Métodos auxiliares para la moda real (MO1 = moda menor en empate, MO2 = moda mayor en empate) ---\n    def _compute_mode(self, x_stat, high=False):\n        # Cuantizamos a niveles enteros de 8 bits (Escala [0, 255] del histograma del paper)\n        q = torch.round(x_stat * 255.0)\n        n_elem = q.size(-1)\n\n        if n_elem \u003c= 64:\n            sorted_q, _ = torch.sort(q, dim=-1, descending=high)\n            counts = torch.zeros_like(sorted_q, dtype=torch.int32)\n            for k in range(n_elem):\n                counts += (sorted_q == sorted_q[..., k:k + 1]).int()\n            best_idx = torch.argmax(counts, dim=-1, keepdim=True)\n            return torch.gather(sorted_q, dim=-1, index=best_idx) / 255.0\n\n        # Caso dominios grandes (sub-regiones W4..W49 o global W1): histograma vectorizado con scatter_add_\n        q_min = q.min()\n        q_span = int((q.max() - q_min).item()) + 1\n        if q_span \u003c= 4096:\n            q_idx = (q - q_min).to(torch.long)\n            hist = torch.zeros((*q.shape[:-1], q_span), dtype=torch.int32, device=x_stat.device)\n            hist.scatter_add_(-1, q_idx, torch.ones_like(q_idx, dtype=torch.int32))\n            if high:\n                best_bin = (q_span - 1) - torch.argmax(hist.flip(-1), dim=-1, keepdim=True)\n            else:\n                best_bin = torch.argmax(hist, dim=-1, keepdim=True)\n            return (best_bin.to(x_stat.dtype) + q_min) / 255.0\n\n        # Fallback general en caso de rangos extremos fuera de escala\n        flat_q = q.reshape(-1, n_elem)\n        out_flat = torch.empty((flat_q.size(0), 1), dtype=x_stat.dtype, device=x_stat.device)\n        for idx in range(flat_q.size(0)):\n            vals, cnts = torch.unique(flat_q[idx], sorted=True, return_counts=True)\n            if high:\n                vals = vals.flip(0)\n                cnts = cnts.flip(0)\n            out_flat[idx, 0] = vals[torch.argmax(cnts)] / 255.0\n        return out_flat.reshape(*q.shape[:-1], 1)\n\n    def _quantize_255(self, x_stat):\n        \"\"\"\n        Convierte las intensidades de x_stat a niveles enteros discretos {0, 1, ..., 255}\n        del histograma por componente RGB. Si el tensor proviene de normalización ImageNet\n        (modelos preentrenados con valores fuera de [0, 1]), restituye primero la escala [0, 1].\n        \"\"\"\n        x_min = float(x_stat.min().item())\n        x_max = float(x_stat.max().item())\n        if (x_min \u003c -1e-4 or x_max \u003e 1.0 + 1e-4) and x_stat.size(1) == 3:\n            shape = [1, 3] + [1] * (x_stat.ndim - 2)\n            mean = torch.tensor([0.485, 0.456, 0.406], dtype=x_stat.dtype, device=x_stat.device).view(*shape)\n            std = torch.tensor([0.229, 0.224, 0.225], dtype=x_stat.dtype, device=x_stat.device).view(*shape)\n            x_01 = (x_stat * std + mean).clamp(0.0, 1.0)\n        else:\n            x_01 = x_stat.clamp(0.0, 1.0)\n        return torch.round(x_01 * 255.0).to(torch.long)\n\n    def _compute_entropy(self, x_stat):\n        \"\"\"\n        Calcula la Entropía de Shannon phi_Ent sobre el histograma de 8 bits (niveles 0 a 255)\n        para cada canal RGB en el dominio D_m (ventana local, sub-región o imagen global):\n            phi_Ent(h_{f_j}^{D_m}) = - sum_{i=0..255, p_j(i)\u003e0} p_j(i) * log2(p_j(i))\n        donde p_j(i) = h_{f_j}^{D_m}(i) / n.\n        \"\"\"\n        q = self._quantize_255(x_stat)\n        n_elem = q.size(-1)\n\n        if n_elem \u003c= 64:\n            # Identidad exacta sin alojar 256 bins por cada ventana espacial:\n            # -sum_{v} (c_v / n) log2(c_v / n) == log2(n) - (1/n) * sum_{m=1..n} log2(c(q_m))\n            counts = torch.zeros(q.shape, dtype=torch.float32, device=x_stat.device)\n            for k in range(n_elem):\n                counts += (q == q[..., k:k + 1]).to(torch.float32)\n            ent = math.log2(float(n_elem)) - torch.mean(torch.log2(counts), dim=-1, keepdim=True)\n            return ent.clamp_min(0.0).to(x_stat.dtype)\n\n        # Para dominios amplios (sub-regiones W4..W64 o global W1): histograma exacto de 256 niveles (0..255)\n        hist = torch.zeros((*q.shape[:-1], 256), dtype=torch.float32, device=x_stat.device)\n        hist.scatter_add_(-1, q, torch.ones_like(q, dtype=torch.float32))\n        probs = hist / float(n_elem)\n        log_probs = torch.where(probs \u003e 0.0, torch.log2(probs.clamp_min(1e-12)), torch.zeros_like(probs))\n        ent = -torch.sum(probs * log_probs, dim=-1, keepdim=True)\n        return ent.clamp_min(0.0).to(x_stat.dtype)\n\n    def _mode_low(self, x_stat):\n        return self._compute_mode(x_stat, high=False)\n\n    def _mode_high(self, x_stat):\n        return self._compute_mode(x_stat, high=True)\n","models.py":"import torch\r\nimport torch.nn as nn\r\nfrom morph_pool_layer import AdaptiveMorphPooling2d\r\nimport torchvision.models as models\r\n\r\nclass CNNClassifier(nn.Module):\r\n    def __init__(self, use_morph_pool=False, metric=\u0027SMO\u0027, stat_window=None, pool_type=\u0027max\u0027, partition=\u0027W0\u0027):\r\n        super().__init__()\r\n        self.use_morph_pool = use_morph_pool\r\n        if use_morph_pool:\r\n            self.pool = AdaptiveMorphPooling2d(2, 2, metric=metric, stat_window=stat_window, partition=partition)\r\n        elif pool_type == \u0027avg\u0027:\r\n            self.pool = nn.AvgPool2d(2, 2)\r\n        else:\r\n            self.pool = nn.MaxPool2d(2, 2)\r\n        self.morph_pool = self.pool\r\n        self.max_pool_rgb = self.pool\r\n\r\n        # Bloques Convolucionales\r\n        self.features = nn.Sequential(\r\n            nn.Conv2d(3, 32, kernel_size=3, padding=1),\r\n            nn.BatchNorm2d(32),\r\n            nn.ReLU(),\r\n            nn.MaxPool2d(2, 2),\r\n            \r\n            nn.Conv2d(32, 64, kernel_size=3, padding=1),\r\n            nn.BatchNorm2d(64),\r\n            nn.ReLU(),\r\n            nn.MaxPool2d(2, 2)\r\n        )\r\n        \r\n        # Clasificador\r\n        self.classifier = nn.Sequential(\r\n            nn.AdaptiveAvgPool2d((1, 1)),\r\n            nn.Flatten(),\r\n            nn.Linear(64, 32),\r\n            nn.ReLU(),\r\n            nn.Linear(32, 1)\r\n        )\r\n\r\n    def forward(self, x):\r\n        x = self.pool(x)\r\n        x = self.features(x)\r\n        return self.classifier(x)\r\n\r\n\r\nclass PretrainedWithPool(nn.Module):\r\n    def __init__(self, backbone_name=\u0027resnet18\u0027, use_morph=False, metric=\u0027SMO\u0027, stat_window=None, pool_type=\u0027max\u0027, partition=\u0027W0\u0027):\r\n        super().__init__()\r\n        self.backbone_name = backbone_name\r\n        if use_morph:\r\n            self.pool = AdaptiveMorphPooling2d(2, 2, metric=metric, stat_window=stat_window, partition=partition)\r\n        elif pool_type == \u0027avg\u0027:\r\n            self.pool = nn.AvgPool2d(2, 2)\r\n        else:\r\n            self.pool = nn.MaxPool2d(2, 2)\r\n\r\n        if backbone_name == \u0027vgg16\u0027:\r\n            bb = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)\r\n            for p in bb.parameters(): p.requires_grad = False\r\n            for i in range(17, len(bb.features)):\r\n                for p in bb.features[i].parameters(): p.requires_grad = True\r\n            self.frozen_features = bb.features[:17].eval()\r\n            self.trainable_features = bb.features[17:]\r\n            self.features = bb.features\r\n            self.avgpool = nn.AdaptiveAvgPool2d((7, 7))\r\n            self.classifier = nn.Sequential(nn.Linear(512*7*7, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 1))\r\n        elif backbone_name == \u0027resnet18\u0027:\r\n            bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)\r\n            for p in bb.parameters(): p.requires_grad = False\r\n            for p in bb.layer3.parameters(): p.requires_grad = True\r\n            for p in bb.layer4.parameters(): p.requires_grad = True\r\n            self.frozen_features = nn.Sequential(\r\n                bb.conv1, bb.bn1, bb.relu, bb.maxpool, bb.layer1, bb.layer2\r\n            ).eval()\r\n            self.trainable_features = nn.Sequential(bb.layer3, bb.layer4)\r\n            self.features = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool,\r\n                                          bb.layer1, bb.layer2, bb.layer3, bb.layer4)\r\n            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))\r\n            self.classifier = nn.Linear(512, 1)\r\n\r\n    def train(self, mode=True):\r\n        super().train(mode)\r\n        self.frozen_features.eval()\r\n        return self\r\n\r\n    @torch.inference_mode()\r\n    def extract_frozen(self, x):\r\n        return self.frozen_features(self.pool(x))\r\n\r\n    def forward_trainable(self, z):\r\n        return self.classifier(torch.flatten(self.avgpool(self.trainable_features(z)), 1))\r\n\r\n    def forward(self, x):\r\n        with torch.no_grad():\r\n            z = self.frozen_features(self.pool(x))\r\n        return self.forward_trainable(z)","train_eval.py":"import os\nimport gc\nimport math\nimport time\nimport random\nfrom concurrent.futures import ThreadPoolExecutor\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom torch.utils.data import Dataset\nfrom torchvision import datasets, transforms\nfrom sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, roc_curve\n\nimport argparse\nfrom morph_pool_layer import normalize_partition, format_domain_label\nfrom models import CNNClassifier, PretrainedWithPool\nfrom report_generator import generate_html_report\nfrom report_aggregator import aggregate_report\n\n\ndef set_seed(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description=\"Experimento A/B: Pooling Morfológico Adaptativo vs MaxPool2d en Imágenes Dermatoscópicas (Melanoma)\"\n    )\n    parser.add_argument(\n        \"-e\", \"--epochs\",\n        type=int,\n        default=5,\n        help=\"Número de etapas/épocas de entrenamiento por modelo (default: 5)\"\n    )\n    parser.add_argument(\n        \"-b\", \"--batch-size\",\n        type=int,\n        default=None,\n        help=\"Tamaño del lote / batch size (default automático según VRAM y arquitectura)\"\n    )\n    parser.add_argument(\n        \"--lr\",\n        type=float,\n        default=1e-3,\n        help=\"Tasa de aprendizaje / Learning rate (default: 0.001 en simple, 0.0001 en preentrenadas)\"\n    )\n    parser.add_argument(\n        \"--metric\",\n        choices=[\"SMO\", \"VAR\", \"ENT\", \"MEAN\", \"MIN\", \"MAX\", \"MO1\", \"MO2\"],\n        default=\"SMO\",\n        help=\"Descriptor estadístico phi por canal RGB (incluye ENT = Entropía 0-255; default: SMO)\"\n    )\n    parser.add_argument(\n        \"-d\", \"--data-dir\", \"--dataset\",\n        dest=\"data_dir\",\n        type=str,\n        default=\"archive\",\n        help=\"Carpeta del dataset a utilizar: \u0027archive\u0027, \u0027archive2\u0027, \u0027archive11k_2\u0027, etc. (default: archive)\"\n    )\n    parser.add_argument(\n        \"-s\", \"--seed\",\n        type=int,\n        default=42,\n        help=\"Semilla aleatoria para reproducibilidad A/B (default: 42)\"\n    )\n    parser.add_argument(\n        \"-o\", \"--output\",\n        type=str,\n        default=\"reporte_experimento_melanoma.html\",\n        help=\"Nombre o ruta del informe HTML de salida; se guarda en \u0027reportes/\u0027 (default: reporte_experimento_melanoma.html)\"\n    )\n    parser.add_argument(\n        \"--stat-window\",\n        type=str,\n        default=\"2\",\n        choices=[\"2\", \"3\", \"5\", \"7\", \"global\"],\n        help=\"Tamaño de la ventana B para vecindad (W0) o ventana de contacto en sub-regiones (WX) (default: 2)\"\n    )\n    parser.add_argument(\n        \"-p\", \"--partition\", \"--domain\",\n        dest=\"partition\",\n        type=str,\n        default=\"W0\",\n        help=(\n            \"Estrategia de partición de dominio según el paper (Sección II-A): \"\n            \"W0 (vecindad local), W1 (global 1x1), W4 (2x2), W9 (3x3), \"\n            \"W16 (4x4), W25 (5x5), W49 (7x7) o MxN (default: W0)\"\n        )\n    )\n    parser.add_argument(\n        \"--model\",\n        type=str,\n        default=\"simple\",\n        choices=[\"simple\", \"vgg16\", \"resnet18\"],\n        help=\"Arquitectura: simple (CNN), vgg16 o resnet18 (default: simple)\"\n    )\n    parser.add_argument(\n        \"--pool\",\n        type=str,\n        default=\"max\",\n        choices=[\"max\", \"avg\"],\n        help=\"Pooling baseline (default: max). El propuesto siempre es morph.\"\n    )\n    parser.add_argument(\n        \"--seeds\",\n        type=int,\n        nargs=\"+\",\n        default=None,\n        help=\"Lista de semillas para ejecutar múltiples experimentos (ej: --seeds 42 7 123)\"\n    )\n    parser.add_argument(\n        \"--epochs-list\",\n        type=int,\n        nargs=\"+\",\n        default=None,\n        help=\"Lista de épocas para ejecutar múltiples experimentos (ej: --epochs-list 5 10)\"\n    )\n    parser.add_argument(\n        \"--amp\",\n        dest=\"amp\",\n        action=\"store_true\",\n        default=None,\n        help=\"Activa precisión mixta FP16 (Tensor Cores en Tesla T4 / GPUs modernas). Por defecto automático si VRAM \u003e= 8 GB.\"\n    )\n    parser.add_argument(\n        \"--no-amp\",\n        dest=\"amp\",\n        action=\"store_false\",\n        help=\"Desactiva precisión mixta FP16 forzando FP32 puro.\"\n    )\n\n    return parser.parse_args()\n\n\n# 1. Dataset precargado en RAM con decodificación JPEG paralela multihilo\nclass FastRAMDataset(Dataset):\n    def __init__(self, folder_path, transform):\n        print(f\"Cargando {folder_path} directamente a memoria RAM...\")\n        raw_dataset = datasets.ImageFolder(folder_path)\n        samples = raw_dataset.samples\n        loader = raw_dataset.loader\n        total = len(samples)\n\n        first_img = transform(loader(samples[0][0]))\n        self.images = torch.empty((total, *first_img.shape), dtype=first_img.dtype)\n        self.labels = torch.empty(total, dtype=torch.float32)\n        self.images[0] = first_img\n        self.labels[0] = samples[0][1]\n\n        def _load_one(idx):\n            path, label = samples[idx]\n            return idx, transform(loader(path)), label\n\n        num_workers = min(8, max(2, (os.cpu_count() or 2) * 2))\n        done_count = 1\n        if total \u003e 1:\n            with ThreadPoolExecutor(max_workers=num_workers) as ex:\n                for idx, t_img, label in ex.map(_load_one, range(1, total), chunksize=32):\n                    self.images[idx] = t_img\n                    self.labels[idx] = label\n                    done_count += 1\n                    if done_count % 500 == 0 or done_count == total:\n                        print(f\"\\r    Leídas {done_count}/{total} imágenes...\", end=\"\", flush=True)\n        print(f\"\\r    Leídas {total}/{total} imágenes.          \")\n        print(f\"--\u003e {total} imágenes listas en RAM.\")\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, idx):\n        return self.images[idx], self.labels[idx]\n\n\n# Detección automática de dispositivo y VRAM disponible\ndef get_best_device():\n    if torch.cuda.is_available():\n        torch.backends.cudnn.benchmark = True\n        if hasattr(torch, \"set_float32_matmul_precision\"):\n            torch.set_float32_matmul_precision(\"high\")\n        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\n        return torch.device(\"cuda\"), f\"CUDA ({torch.cuda.get_device_name(0)} - {vram_gb:.1f} GB VRAM)\", vram_gb\n    if hasattr(torch, \"xpu\") and torch.xpu.is_available():\n        return torch.device(\"xpu\"), \"Intel XPU\", 0.0\n    return torch.device(\"cpu\"), f\"CPU ({torch.get_num_threads()} hilos con oneDNN/MKLDNN)\", 0.0\n\n\nDEVICE, DEVICE_NAME, TOTAL_VRAM_GB = get_best_device()\n\ntransform_simple = transforms.Compose([\n    transforms.Resize((112, 112)),\n    transforms.ToTensor()\n])\n\ntransform_pretrained = transforms.Compose([\n    transforms.Resize((224, 224)),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.485, 0.456, 0.406],\n                         std=[0.229, 0.224, 0.225])\n])\n\n\ndef _can_store_in_vram(needed_bytes, reserve_gb=1.8):\n    \"\"\"Verifica si el tensor precalculado cabe holgadamente directo en la VRAM de la GPU.\"\"\"\n    if DEVICE.type != \"cuda\":\n        return False\n    try:\n        free_bytes, _ = torch.cuda.mem_get_info()\n        return (free_bytes - needed_bytes) \u003e int(reserve_gb * (1024 ** 3))\n    except Exception:\n        return False\n\n\n@torch.inference_mode()\ndef _precompute_static_features(model, model_name, images_tensor, batch_size, desc=\"\", use_amp=False):\n    \"\"\"\n    Precalcula una sola vez la etapa estática/congelada (pooling inicial en FP32 exacto + bloques congelados).\n    Si hay suficiente VRAM libre (como los 15 GB de Tesla T4), aloja el tensor directamente en VRAM\n    para eliminar al 100% las transferencias CPU-\u003eGPU durante todas las épocas y semillas.\n    \"\"\"\n    model.eval()\n    out_tensor = None\n    target_dev = torch.device(\"cpu\")\n    n = images_tensor.size(0)\n    # En inferencia estática sin gradientes podemos usar lotes grandes para exprimir la GPU\n    if DEVICE.type == \"cuda\" and TOTAL_VRAM_GB \u003e= 8.0:\n        step = max(batch_size, 128)\n    elif DEVICE.type == \"cuda\":\n        step = max(batch_size, 32)\n    else:\n        step = max(batch_size, 64)\n\n    for i in range(0, n, step):\n        batch = images_tensor[i:i + step].to(DEVICE, non_blocking=True)\n        if model_name == \"simple\":\n            # El pooling RGB siempre corre en FP32 exacto para máxima precisión estadística\n            out = model.morph_pool(batch) if model.use_morph_pool else model.max_pool_rgb(batch)\n        else:\n            # El pooling RGB corre en FP32 exacto; las convoluciones congeladas pueden usar Tensor Cores\n            pooled = model.pool(batch)\n            if use_amp and DEVICE.type == \"cuda\":\n                with torch.amp.autocast(\"cuda\", dtype=torch.float16):\n                    out = model.frozen_features(pooled)\n            else:\n                out = model.frozen_features(pooled)\n\n        if out_tensor is None:\n            needed_bytes = n * math.prod(out.shape[1:]) * out.element_size()\n            if _can_store_in_vram(needed_bytes):\n                target_dev = DEVICE\n                out_tensor = torch.empty((n, *out.shape[1:]), dtype=out.dtype, device=DEVICE)\n            else:\n                target_dev = torch.device(\"cpu\")\n                out_tensor = torch.empty((n, *out.shape[1:]), dtype=out.dtype, pin_memory=(DEVICE.type == \"cuda\"))\n\n        if target_dev.type == \"cuda\":\n            out_tensor[i:i + out.size(0)].copy_(out)\n        else:\n            out_tensor[i:i + out.size(0)].copy_(out.cpu())\n\n        if (i // step) % 4 == 0 or (i + step) \u003e= n:\n            loc_str = \"VRAM GPU\" if target_dev.type == \"cuda\" else \"RAM CPU\"\n            print(f\"\\r    [{desc} -\u003e {loc_str}] Procesadas {min(i + step, n)}/{n} imágenes...\", end=\"\", flush=True)\n    print()\n    return out_tensor\n\n\ndef _build_model(model_name, pool_type, use_morph, metric, stat_window, partition):\n    if model_name == \"simple\":\n        return CNNClassifier(\n            use_morph_pool=use_morph, metric=metric,\n            stat_window=stat_window, pool_type=pool_type, partition=partition\n        ).to(DEVICE)\n    return PretrainedWithPool(\n        model_name, use_morph=use_morph, metric=metric,\n        stat_window=stat_window, pool_type=pool_type, partition=partition\n    ).to(DEVICE)\n\n\ndef run_experiment(model_name, pool_type, use_morph, name, epochs, batch_size, lr,\n                   metric, stat_window, train_dataset, test_dataset, seed=42,\n                   partition=\"W0\", use_amp=False, precomputed=None):\n    set_seed(seed)\n    model = _build_model(model_name, pool_type, use_morph, metric, stat_window, partition)\n\n    total_params = sum(p.numel() for p in model.parameters())\n    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    amp_tag = \"FP16 TensorCores\" if (use_amp and DEVICE.type == \"cuda\") else \"FP32\"\n    print(f\"[Parámetros {model_name} | {amp_tag}] Entrenables: {trainable_params:,} / Total: {total_params:,}\")\n\n    criterion = nn.BCEWithLogitsLoss()\n    use_foreach = DEVICE.type in (\"cuda\", \"cpu\")\n    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, foreach=use_foreach)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=(use_amp and DEVICE.type == \"cuda\"))\n\n    print(f\"\\n================ Ejecutando: {name} ({epochs} épocas | Seed: {seed}) ================\")\n    start_time = time.time()\n\n    own_precomputed = False\n    if precomputed is not None:\n        train_x, test_x = precomputed\n        loc_label = \"VRAM GPU\" if train_x.is_cuda else \"RAM\"\n        print(f\" -\u003e Reutilizando características estáticas ya alojadas en {loc_label} (0.0s overhead).\")\n    else:\n        own_precomputed = True\n        print(\" -\u003e Precalculando etapa estática (Pooling RGB + bloques congelados)...\")\n        train_x = _precompute_static_features(model, model_name, train_dataset.images, batch_size, desc=\"Train\", use_amp=use_amp)\n        test_x = _precompute_static_features(model, model_name, test_dataset.images, batch_size, desc=\"Test\", use_amp=use_amp)\n\n    train_y = train_dataset.labels.to(DEVICE) if train_x.is_cuda else train_dataset.labels\n    test_y_cpu = test_dataset.labels\n    train_size = len(train_dataset.labels)\n    test_size = len(test_y_cpu)\n    x_in_vram = train_x.is_cuda\n\n    # Reiniciar semilla antes del bucle de épocas para garantizar idéntico barajado A/B\n    set_seed(seed)\n    epoch_losses = []\n    num_batches = max(1, (train_size + batch_size - 1) // batch_size)\n    log_every = max(1, num_batches // 4)\n\n    for epoch in range(epochs):\n        ep_t0 = time.time()\n        model.train()\n        running_loss_t = torch.zeros((), dtype=torch.float32, device=DEVICE)\n        # Mantener torch.randperm en CPU preserva exactamente la misma secuencia de lotes entre CPU/GPU\n        perm = torch.randperm(train_size)\n        if x_in_vram:\n            perm_dev = perm.to(DEVICE, non_blocking=True)\n\n        for b_idx, i in enumerate(range(0, train_size, batch_size)):\n            if x_in_vram:\n                idx = perm_dev[i:i + batch_size]\n                images = train_x[idx]\n                labels = train_y[idx].unsqueeze(1)\n            else:\n                idx = perm[i:i + batch_size]\n                images = train_x[idx].to(DEVICE, non_blocking=True)\n                labels = train_y[idx].unsqueeze(1).to(DEVICE, non_blocking=True)\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.amp.autocast(\"cuda\", dtype=torch.float16, enabled=(use_amp and DEVICE.type == \"cuda\")):\n                if model_name == \"simple\":\n                    outputs = model.classifier(model.features(images))\n                else:\n                    outputs = model.forward_trainable(images)\n                loss = criterion(outputs, labels)\n\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n\n            running_loss_t += loss.detach().float() * images.size(0)\n            if (b_idx + 1) % log_every == 0 or (b_idx + 1) == num_batches:\n                print(f\"\\r    Época [{epoch+1}/{epochs}] Lote {min(i + batch_size, train_size)}/{train_size}...\", end=\"\", flush=True)\n\n        epoch_loss = (running_loss_t / train_size).item()\n        epoch_losses.append(epoch_loss)\n        print(f\"\\rÉpoca [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} ({time.time() - ep_t0:.2f}s)          \")\n\n    elapsed = time.time() - start_time\n    print(f\"Tiempo de entrenamiento: {elapsed:.2f} segundos.\")\n\n    # Evaluación en Test\n    model.eval()\n    all_probs = []\n    eval_step = max(batch_size, 128) if (DEVICE.type == \"cuda\" and TOTAL_VRAM_GB \u003e= 8.0) else batch_size\n    with torch.inference_mode():\n        for i in range(0, test_size, eval_step):\n            images = test_x[i:i + eval_step] if test_x.is_cuda else test_x[i:i + eval_step].to(DEVICE, non_blocking=True)\n            with torch.amp.autocast(\"cuda\", dtype=torch.float16, enabled=(use_amp and DEVICE.type == \"cuda\")):\n                if model_name == \"simple\":\n                    outputs = model.classifier(model.features(images))\n                else:\n                    outputs = model.forward_trainable(images)\n            probs = torch.sigmoid(outputs.float())\n            all_probs.append(probs.cpu())\n\n    all_probs = torch.cat(all_probs, dim=0).numpy().ravel()\n    all_preds = (all_probs \u003e 0.5).astype(np.float32)\n    all_labels = test_y_cpu.numpy().ravel()\n\n    acc = accuracy_score(all_labels, all_preds)\n    sens = recall_score(all_labels, all_preds)\n    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()\n    spec = tn / (tn + fp)\n    auc = roc_auc_score(all_labels, all_probs)\n    fpr, tpr, _ = roc_curve(all_labels, all_probs)\n\n    print(f\"\\n--- Resultados Finales ({name}) ---\")\n    print(f\"Accuracy:      {acc*100:.2f}%\")\n    print(f\"Sensibilidad:  {sens*100:.2f}%\")\n    print(f\"Especificidad: {spec*100:.2f}%\")\n    print(f\"AUC-ROC:       {auc:.4f}\")\n\n    del model\n    if own_precomputed:\n        del train_x, test_x\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    return {\n        \"name\": name,\n        \"use_morph\": use_morph,\n        \"epoch_losses\": epoch_losses,\n        \"elapsed\": elapsed,\n        \"acc\": acc,\n        \"sens\": sens,\n        \"spec\": spec,\n        \"auc\": auc,\n        \"tn\": int(tn),\n        \"fp\": int(fp),\n        \"fn\": int(fn),\n        \"tp\": int(tp),\n        \"fpr\": fpr.tolist(),\n        \"tpr\": tpr.tolist()\n    }\n\n\nif __name__ == \u0027__main__\u0027:\n    args = parse_args()\n    # Si hay \u003e= 8 GB de VRAM (ej. Tesla T4 15 GB en Colab), usamos batch 64 por defecto también en ResNet/VGG;\n    # en GPUs de 4 GB (GTX 1650) mantenemos 32/16 para cuidar VRAM salvo que el usuario indique -b.\n    if DEVICE.type == \"cuda\" and TOTAL_VRAM_GB \u003c 7.5:\n        default_bs = {\"simple\": 64, \"resnet18\": 32, \"vgg16\": 16}\n    else:\n        default_bs = {\"simple\": 64, \"resnet18\": 64, \"vgg16\": 64}\n\n    batch_size = args.batch_size if args.batch_size is not None else default_bs[args.model]\n    lr = args.lr if args.model == \"simple\" else 1e-4\n    use_amp = args.amp if args.amp is not None else (DEVICE.type == \"cuda\" and TOTAL_VRAM_GB \u003e= 7.5)\n    selected_transform = transform_simple if args.model == \"simple\" else transform_pretrained\n    resolution = \"112x112\" if args.model == \"simple\" else \"224x224\"\n    stat_window = args.stat_window if args.stat_window == \"global\" else int(args.stat_window)\n    try:\n        partition_norm = normalize_partition(args.partition, stat_window)\n    except ValueError as exc:\n        print(f\"\\n[!] Error en argumento --partition: {exc}\\n\")\n        exit(1)\n    domain_label = format_domain_label(partition_norm, stat_window)\n    short_domain_tag = domain_label.split(\" \")[0]\n\n    raw_dir = args.data_dir.strip()\n    candidates_map = {\n        \"1\": [\"archive\", \"archive1\", \"archive 1\"],\n        \"archive\": [\"archive\", \"archive1\", \"archive 1\"],\n        \"archive1\": [\"archive\", \"archive1\", \"archive 1\"],\n        \"archive 1\": [\"archive\", \"archive1\", \"archive 1\"],\n        \"2\": [\"archive2\", \"archive 2\"],\n        \"archive2\": [\"archive2\", \"archive 2\"],\n        \"archive 2\": [\"archive2\", \"archive 2\"],\n        \"3\": [\"archive3\", \"archive 3\"],\n        \"archive3\": [\"archive3\", \"archive 3\"],\n        \"archive 3\": [\"archive3\", \"archive 3\"],\n    }\n    candidates = candidates_map.get(raw_dir.lower(), [raw_dir, raw_dir.replace(\" \", \"\")])\n    data_dir = next((c for c in candidates if os.path.exists(c)), candidates[0])\n    dataset_label = os.path.basename(os.path.normpath(data_dir))\n\n    seeds_list = args.seeds if args.seeds is not None else [args.seed]\n    epochs_list = args.epochs_list if args.epochs_list is not None else [args.epochs]\n    multi_run = (len(seeds_list) \u003e 1 or len(epochs_list) \u003e 1)\n\n    print(\"\\n\" + \"=\"*65)\n    print(f\" INICIO DEL EXPERIMENTO COMPARATIVO A/B (MELANOMA) - {args.model.upper()}\")\n    print(f\" Dataset: {dataset_label} | Épocas: {epochs_list} | Batch Size: {batch_size} | LR: {lr}\")\n    print(f\" Métrica: {args.metric} | Dominio: {domain_label} | Pool Baseline: {args.pool.upper()} | Seeds: {seeds_list}\")\n    print(f\" Dispositivo: {DEVICE_NAME} | Precisión: {\u0027AMP FP16 (Tensor Cores)\u0027 if use_amp else \u0027FP32\u0027}\")\n    print(\"=\"*65)\n\n    if args.model != \"simple\" and args.metric not in (\"SMO\", \"VAR\", \"ENT\"):\n        print(f\"\\n[!] ADVERTENCIA METODOLÓGICA: \u0027{args.metric}\u0027 se evalúa sobre tensores normalizados con ImageNet\")\n        print(\"    (valores centrados en 0 y negativos). Se recomienda usar \u0027SMO\u0027, \u0027VAR\u0027 o \u0027ENT\u0027 para\")\n        print(\"    comparaciones consistentes entre arquitecturas.\\n\")\n\n    train_dir = os.path.join(data_dir, \"train\")\n    test_dir = os.path.join(data_dir, \"test\")\n\n    if not os.path.exists(train_dir) or not os.path.exists(test_dir):\n        print(f\"\\n[!] Error: No se encontraron las carpetas \u0027train\u0027 o \u0027test\u0027 en: {data_dir}\")\n        print(\"    Ejemplos de uso: -d archive | -d \\\"archive 2\\\" | -d \\\"archive 3\\\"\\n\")\n        exit(1)\n\n    print(\"\\n=== PREPARANDO DATASETS EN RAM (Precarga paralela) ===\")\n    train_dataset = FastRAMDataset(train_dir, transform=selected_transform)\n    test_dataset = FastRAMDataset(test_dir, transform=selected_transform)\n\n    # Precalcular UNA SOLA VEZ las características estáticas para Baseline y MorphPool\n    # (son deterministas e independientes de la semilla, ahorrando minutos enormes en multi-seed)\n    print(\"\\n=== PRECALCULANDO CARACTERÍSTICAS ESTÁTICAS EN VRAM/RAM (1 sola vez para todas las semillas) ===\")\n    t_pre0 = time.time()\n    tmp_base_model = _build_model(args.model, args.pool, False, args.metric, stat_window, partition_norm)\n    base_train_x = _precompute_static_features(tmp_base_model, args.model, train_dataset.images, batch_size, desc=\"Baseline Train\", use_amp=use_amp)\n    base_test_x = _precompute_static_features(tmp_base_model, args.model, test_dataset.images, batch_size, desc=\"Baseline Test\", use_amp=use_amp)\n    del tmp_base_model\n\n    tmp_prop_model = _build_model(args.model, \"morph\", True, args.metric, stat_window, partition_norm)\n    prop_train_x = _precompute_static_features(tmp_prop_model, args.model, train_dataset.images, batch_size, desc=\"MorphPool Train\", use_amp=use_amp)\n    prop_test_x = _precompute_static_features(tmp_prop_model, args.model, test_dataset.images, batch_size, desc=\"MorphPool Test\", use_amp=use_amp)\n    del tmp_prop_model\n\n    # Liberar los tensores crudos 224x224 de la RAM de sistema (libera hasta 6.6 GB en Colab!)\n    train_dataset.images = None\n    test_dataset.images = None\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        used_vram = torch.cuda.memory_allocated() / (1024 ** 3)\n        print(f\"--\u003e Precomputación completada en {time.time() - t_pre0:.1f}s | VRAM activa con tensores cacheados: {used_vram:.2f} GB\")\n\n    base_out_name, base_out_ext = os.path.splitext(args.output)\n    if not base_out_ext:\n        base_out_ext = \".html\"\n\n    for ep in epochs_list:\n        ep_results = []\n        for seed in seeds_list:\n            set_seed(seed)\n            baseline_results = run_experiment(\n                model_name=args.model,\n                pool_type=args.pool,\n                use_morph=False,\n                name=f\"Baseline ({args.model.upper()} + {args.pool.upper()}Pool RGB)\",\n                epochs=ep,\n                batch_size=batch_size,\n                lr=lr,\n                metric=args.metric,\n                stat_window=stat_window,\n                train_dataset=train_dataset,\n                test_dataset=test_dataset,\n                seed=seed,\n                partition=partition_norm,\n                use_amp=use_amp,\n                precomputed=(base_train_x, base_test_x)\n            )\n\n            proposed_results = run_experiment(\n                model_name=args.model,\n                pool_type=\"morph\",\n                use_morph=True,\n                name=f\"Propuesto ({args.model.upper()} + MorphPool {args.metric} {short_domain_tag})\",\n                epochs=ep,\n                batch_size=batch_size,\n                lr=lr,\n                metric=args.metric,\n                stat_window=stat_window,\n                train_dataset=train_dataset,\n                test_dataset=test_dataset,\n                seed=seed,\n                partition=partition_norm,\n                use_amp=use_amp,\n                precomputed=(prop_train_x, prop_test_x)\n            )\n\n            run_output = (\n                f\"{base_out_name}_s{seed}_e{ep}{base_out_ext}\"\n                if multi_run else args.output\n            )\n            run_config = {\n                \"dataset\": f\"ISIC ({dataset_label})\",\n                \"model\": args.model,\n                \"pool\": args.pool,\n                \"seed\": seed,\n                \"batch_size\": batch_size,\n                \"epochs\": ep,\n                \"lr\": lr,\n                \"metric\": args.metric,\n                \"stat_window\": args.stat_window,\n                \"partition\": partition_norm,\n                \"domain_label\": domain_label,\n                \"device\": DEVICE_NAME,\n                \"resolution\": resolution,\n                \"train_samples\": len(train_dataset),\n                \"test_samples\": len(test_dataset),\n                \"date\": time.strftime(\"%Y-%m-%d %H:%M:%S\")\n            }\n\n            report_path = generate_html_report(\n                baseline=baseline_results,\n                proposed=proposed_results,\n                config=run_config,\n                output_path=run_output\n            )\n\n            print(\"\\n\" + \"=\"*65)\n            print(f\" [OK] INFORME INDIVIDUAL GENERADO (Seed: {seed} | Épocas: {ep})\")\n            print(f\" Archivo HTML: {report_path}\")\n            print(\"=\"*65)\n\n            ep_results.append({\n                \"seed\": seed,\n                \"epochs\": ep,\n                \"baseline\": baseline_results,\n                \"proposed\": proposed_results\n            })\n\n        if len(seeds_list) \u003e 1:\n            agg_out = f\"{base_out_name}_agregado_e{ep}{base_out_ext}\"\n            agg_path = aggregate_report(\n                results=ep_results,\n                output_path=agg_out,\n                config={\n                    \"dataset\": f\"ISIC ({dataset_label})\",\n                    \"model\": args.model,\n                    \"pool\": args.pool,\n                    \"epochs\": ep,\n                    \"metric\": args.metric,\n                    \"stat_window\": args.stat_window,\n                    \"partition\": partition_norm,\n                    \"domain_label\": domain_label,\n                    \"device\": DEVICE_NAME,\n                }\n            )\n            print(\"\\n\" + \"=\"*65)\n            print(f\" [OK] INFORME ESTADÍSTICO AGREGADO ({len(seeds_list)} seeds | Épocas: {ep})\")\n            print(f\" Archivo HTML Agregado: {agg_path}\")\n            print(\"=\"*65 + \"\\n\")\n","report_generator.py":"import os\r\nimport json\r\nimport time\r\n\r\ndef _svg_loss_curves(losses_a, losses_b, epochs):\r\n    \"\"\"Genera un gráfico SVG vectorial de la evolución de la pérdida por época.\"\"\"\r\n    width, height = 520, 280\r\n    pad_left, pad_right, pad_top, pad_bottom = 50, 25, 30, 45\r\n    plot_w = width - pad_left - pad_right\r\n    plot_h = height - pad_top - pad_bottom\r\n\r\n    all_losses = losses_a + losses_b\r\n    min_val = max(0.0, min(all_losses) * 0.9)\r\n    max_val = max(all_losses) * 1.05\r\n    val_range = max_val - min_val if max_val \u003e min_val else 1.0\r\n\r\n    def pt(epoch_idx, val):\r\n        x = pad_left + (epoch_idx / max(1, epochs - 1)) * plot_w\r\n        y = pad_top + (1.0 - (val - min_val) / val_range) * plot_h\r\n        return x, y\r\n\r\n    # Ejes y rejilla\r\n    grid_lines = []\r\n    for i in range(5):\r\n        y_val = min_val + (val_range * i / 4)\r\n        y_pos = pad_top + (1.0 - i / 4) * plot_h\r\n        grid_lines.append(f\u0027\u003cline x1=\"{pad_left}\" y1=\"{y_pos:.1f}\" x2=\"{width - pad_right}\" y2=\"{y_pos:.1f}\" stroke=\"#e2e8f0\" stroke-dasharray=\"3,3\" /\u003e\u0027)\r\n        grid_lines.append(f\u0027\u003ctext x=\"{pad_left - 8}\" y=\"{y_pos + 4:.1f}\" text-anchor=\"end\" font-size=\"11\" fill=\"#64748b\" font-family=\"sans-serif\"\u003e{y_val:.3f}\u003c/text\u003e\u0027)\r\n\r\n    x_labels = []\r\n    for e in range(epochs):\r\n        x_pos = pad_left + (e / max(1, epochs - 1)) * plot_w\r\n        x_labels.append(f\u0027\u003cline x1=\"{x_pos:.1f}\" y1=\"{height - pad_bottom}\" x2=\"{x_pos:.1f}\" y2=\"{height - pad_bottom + 5}\" stroke=\"#94a3b8\" /\u003e\u0027)\r\n        x_labels.append(f\u0027\u003ctext x=\"{x_pos:.1f}\" y=\"{height - pad_bottom + 18}\" text-anchor=\"middle\" font-size=\"11\" fill=\"#64748b\" font-family=\"sans-serif\"\u003eÉp {e+1}\u003c/text\u003e\u0027)\r\n\r\n    def make_path(losses, color):\r\n        pts = [pt(i, l) for i, l in enumerate(losses)]\r\n        d = \"M \" + \" L \".join(f\"{x:.1f} {y:.1f}\" for x, y in pts)\r\n        circles = \"\".join(f\u0027\u003ccircle cx=\"{x:.1f}\" cy=\"{y:.1f}\" r=\"4\" fill=\"{color}\" stroke=\"#ffffff\" stroke-width=\"2\" /\u003e\u0027 for x, y in pts)\r\n        return f\u0027\u003cpath d=\"{d}\" fill=\"none\" stroke=\"{color}\" stroke-width=\"3\" stroke-linecap=\"round\" stroke-linejoin=\"round\" /\u003e{circles}\u0027\r\n\r\n    path_a = make_path(losses_a, \"#ef4444\")  # Rojo/ámbar para baseline\r\n    path_b = make_path(losses_b, \"#10b981\")  # Verde/esmeralda para propuesto\r\n\r\n    svg = f\u0027\u0027\u0027\r\n    \u003csvg viewBox=\"0 0 {width} {height}\" class=\"w-full h-auto\" xmlns=\"http://www.w3.org/2000/svg\"\u003e\r\n        \u003c!-- Rejilla --\u003e\r\n        {\u0027\u0027.join(grid_lines)}\r\n        {\u0027\u0027.join(x_labels)}\r\n        \u003c!-- Ejes --\u003e\r\n        \u003cline x1=\"{pad_left}\" y1=\"{pad_top}\" x2=\"{pad_left}\" y2=\"{height - pad_bottom}\" stroke=\"#94a3b8\" stroke-width=\"1.5\" /\u003e\r\n        \u003cline x1=\"{pad_left}\" y1=\"{height - pad_bottom}\" x2=\"{width - pad_right}\" y2=\"{height - pad_bottom}\" stroke=\"#94a3b8\" stroke-width=\"1.5\" /\u003e\r\n        \u003c!-- Curvas --\u003e\r\n        {path_a}\r\n        {path_b}\r\n    \u003c/svg\u003e\r\n    \u0027\u0027\u0027\r\n    return svg\r\n\r\ndef _svg_roc_curves(fpr_a, tpr_a, auc_a, fpr_b, tpr_b, auc_b):\r\n    \"\"\"Genera la curva ROC comparativa en formato vectorial SVG.\"\"\"\r\n    width, height = 400, 320\r\n    pad_left, pad_right, pad_top, pad_bottom = 50, 25, 30, 45\r\n    plot_w = width - pad_left - pad_right\r\n    plot_h = height - pad_top - pad_bottom\r\n\r\n    def pt(fpr, tpr):\r\n        x = pad_left + fpr * plot_w\r\n        y = pad_top + (1.0 - tpr) * plot_h\r\n        return x, y\r\n\r\n    def downsample_path(fpr_list, tpr_list, color):\r\n        # Muestrear si hay demasiados puntos\r\n        step = max(1, len(fpr_list) // 60)\r\n        coords = [pt(fpr_list[i], tpr_list[i]) for i in range(0, len(fpr_list), step)]\r\n        if coords[-1] != pt(fpr_list[-1], tpr_list[-1]):\r\n            coords.append(pt(fpr_list[-1], tpr_list[-1]))\r\n        d = \"M \" + \" L \".join(f\"{x:.1f} {y:.1f}\" for x, y in coords)\r\n        return f\u0027\u003cpath d=\"{d}\" fill=\"none\" stroke=\"{color}\" stroke-width=\"2.5\" stroke-linecap=\"round\" /\u003e\u0027\r\n\r\n    path_a = downsample_path(fpr_a, tpr_a, \"#ef4444\")\r\n    path_b = downsample_path(fpr_b, tpr_b, \"#10b981\")\r\n    diag_line = f\u0027\u003cline x1=\"{pad_left}\" y1=\"{pad_top + plot_h}\" x2=\"{pad_left + plot_w}\" y2=\"{pad_top}\" stroke=\"#cbd5e1\" stroke-dasharray=\"4,4\" stroke-width=\"1.5\" /\u003e\u0027\r\n\r\n    # Ejes y marcas\r\n    ticks = []\r\n    for v in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:\r\n        x_pos = pad_left + v * plot_w\r\n        y_pos = pad_top + (1.0 - v) * plot_h\r\n        ticks.append(f\u0027\u003cline x1=\"{pad_left}\" y1=\"{y_pos:.1f}\" x2=\"{width - pad_right}\" y2=\"{y_pos:.1f}\" stroke=\"#f1f5f9\" /\u003e\u0027)\r\n        ticks.append(f\u0027\u003ctext x=\"{pad_left - 8}\" y=\"{y_pos + 4:.1f}\" text-anchor=\"end\" font-size=\"10\" fill=\"#64748b\"\u003e{v:.1f}\u003c/text\u003e\u0027)\r\n        ticks.append(f\u0027\u003ctext x=\"{x_pos:.1f}\" y=\"{height - pad_bottom + 16}\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\"\u003e{v:.1f}\u003c/text\u003e\u0027)\r\n\r\n    svg = f\u0027\u0027\u0027\r\n    \u003csvg viewBox=\"0 0 {width} {height}\" class=\"w-full h-auto\" xmlns=\"http://www.w3.org/2000/svg\"\u003e\r\n        {\u0027\u0027.join(ticks)}\r\n        {diag_line}\r\n        \u003c!-- Ejes principales --\u003e\r\n        \u003cline x1=\"{pad_left}\" y1=\"{pad_top}\" x2=\"{pad_left}\" y2=\"{height - pad_bottom}\" stroke=\"#94a3b8\" stroke-width=\"1.5\" /\u003e\r\n        \u003cline x1=\"{pad_left}\" y1=\"{height - pad_bottom}\" x2=\"{width - pad_right}\" y2=\"{height - pad_bottom}\" stroke=\"#94a3b8\" stroke-width=\"1.5\" /\u003e\r\n        \u003c!-- Etiquetas ejes --\u003e\r\n        \u003ctext x=\"{width / 2 + 10}\" y=\"{height - 8}\" text-anchor=\"middle\" font-size=\"11\" font-weight=\"600\" fill=\"#475569\"\u003eTasa de Falsos Positivos (1 - Esp)\u003c/text\u003e\r\n        \u003ctext x=\"14\" y=\"{height / 2 - 10}\" text-anchor=\"middle\" font-size=\"11\" font-weight=\"600\" fill=\"#475569\" transform=\"rotate(-90 14 {height / 2 - 10})\"\u003eSensibilidad (TPR)\u003c/text\u003e\r\n        \u003c!-- Curvas ROC --\u003e\r\n        {path_a}\r\n        {path_b}\r\n    \u003c/svg\u003e\r\n    \u0027\u0027\u0027\r\n    return svg\r\n\r\ndef generate_html_report(baseline, proposed, config, output_path=\"reporte_experimento_melanoma.html\"):\r\n    \"\"\"\r\n    Construye un informe científico de primer nivel en HTML con estética moderna,\r\n    gráficos vectoriales interactivos y formateo médico formal para la tesis.\r\n    \"\"\"\r\n    # Cálculos diferenciales\r\n    d_acc = proposed[\"acc\"] - baseline[\"acc\"]\r\n    d_sens = proposed[\"sens\"] - baseline[\"sens\"]\r\n    d_spec = proposed[\"spec\"] - baseline[\"spec\"]\r\n    d_auc = proposed[\"auc\"] - baseline[\"auc\"]\r\n    time_diff = proposed[\"elapsed\"] - baseline[\"elapsed\"]\r\n\r\n    # Generar SVGs\r\n    loss_svg = _svg_loss_curves(baseline[\"epoch_losses\"], proposed[\"epoch_losses\"], len(baseline[\"epoch_losses\"]))\r\n    roc_svg = _svg_roc_curves(\r\n        baseline[\"fpr\"], baseline[\"tpr\"], baseline[\"auc\"],\r\n        proposed[\"fpr\"], proposed[\"tpr\"], proposed[\"auc\"]\r\n    )\r\n\r\n    def badge(diff, is_percentage=True):\r\n        val = diff * 100 if is_percentage else diff\r\n        sym = \"+\" if val \u003e 0 else \"\"\r\n        unit = \"%\" if is_percentage else \"\"\r\n        fmt = f\"{val:.2f}\" if is_percentage else f\"{val:.4f}\"\r\n        cls = \"bg-emerald-100 text-emerald-800 border-emerald-300\" if val \u003e= 0 else \"bg-rose-100 text-rose-800 border-rose-300\"\r\n        return f\u0027\u003cspan class=\"inline-flex items-center px-2 py-0.5 rounded text-xs font-semibold border {cls}\"\u003e{sym}{fmt}{unit}\u003c/span\u003e\u0027\r\n\r\n    # Totales de test\r\n    total_test = baseline[\"tn\"] + baseline[\"fp\"] + baseline[\"fn\"] + baseline[\"tp\"]\r\n\r\n    html_content = f\u0027\u0027\u0027\u003c!DOCTYPE html\u003e\r\n\u003chtml lang=\"es\"\u003e\r\n\u003chead\u003e\r\n    \u003cmeta charset=\"UTF-8\"\u003e\r\n    \u003cmeta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\"\u003e\r\n    \u003ctitle\u003eInforme Científico A/B - Pooling Morfológico Adaptativo (Melanoma) — {config.get(\u0027model\u0027,\u0027simple\u0027).upper()} ({config.get(\u0027pool\u0027,\u0027max\u0027).upper()} vs Morph)\u003c/title\u003e\r\n    \u003c!-- Tailwind CSS (CDN para renderizado impecable de componentes) --\u003e\r\n    \u003cscript src=\"https://cdn.tailwindcss.com\"\u003e\u003c/script\u003e\r\n    \u003clink rel=\"preconnect\" href=\"https://fonts.googleapis.com\"\u003e\r\n    \u003clink rel=\"preconnect\" href=\"https://fonts.gstatic.com\" crossorigin\u003e\r\n    \u003clink href=\"https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800\u0026family=JetBrains+Mono:wght@400;600\u0026display=swap\" rel=\"stylesheet\"\u003e\r\n    \u003cstyle\u003e\r\n        body {{\r\n            font-family: \u0027Inter\u0027, -apple-system, BlinkMacSystemFont, sans-serif;\r\n            background-color: #f8fafc;\r\n            color: #1e293b;\r\n        }}\r\n        .font-mono {{\r\n            font-family: \u0027JetBrains Mono\u0027, monospace;\r\n        }}\r\n        @media print {{\r\n            .no-print {{ display: none !important; }}\r\n            body {{ background-color: #ffffff; }}\r\n            .shadow-sm, .shadow-md, .shadow-lg {{ box-shadow: none !important; }}\r\n            .page-break {{ page-break-before: always; }}\r\n        }}\r\n        .cm-cell {{\r\n            transition: all 0.2s ease;\r\n        }}\r\n        .cm-cell:hover {{\r\n            transform: scale(1.02);\r\n        }}\r\n    \u003c/style\u003e\r\n\u003c/head\u003e\r\n\u003cbody class=\"p-4 md:p-8 max-w-6xl mx-auto\"\u003e\r\n\r\n    \u003c!-- BARRA DE ACCIONES SUPERIOR --\u003e\r\n    \u003cdiv class=\"no-print flex justify-between items-center mb-6 pb-4 border-b border-slate-200\"\u003e\r\n        \u003cdiv class=\"flex items-center space-x-2 text-sm text-slate-500\"\u003e\r\n            \u003cspan class=\"inline-block w-2.5 h-2.5 rounded-full bg-emerald-500 animate-pulse\"\u003e\u003c/span\u003e\r\n            \u003cspan\u003eExperimento A/B Concluido\u003c/span\u003e\r\n            \u003cspan\u003e•\u003c/span\u003e\r\n            \u003cspan class=\"font-mono text-xs\"\u003e{config.get(\u0027date\u0027, time.strftime(\u0027%Y-%m-%d %H:%M:%S\u0027))}\u003c/span\u003e\r\n        \u003c/div\u003e\r\n        \u003cdiv class=\"flex space-x-3\"\u003e\r\n            \u003cbutton onclick=\"window.print()\" class=\"inline-flex items-center px-4 py-2 bg-slate-900 hover:bg-slate-800 text-white text-xs font-semibold rounded-lg shadow-sm transition\"\u003e\r\n                \u003csvg class=\"w-4 h-4 mr-1.5\" fill=\"none\" stroke=\"currentColor\" viewBox=\"0 0 24 24\"\u003e\u003cpath stroke-linecap=\"round\" stroke-linejoin=\"round\" stroke-width=\"2\" d=\"M17 17h2a2 2 0 002-2v-4a2 2 0 00-2-2H5a2 2 0 00-2 2v4a2 2 0 002 2h2m2 4h6a2 2 0 002-2v-4a2 2 0 00-2-2H9a2 2 0 00-2 2v4a2 2 0 002 2zm8-12V5a2 2 0 00-2-2H9a2 2 0 00-2 2v4h10z\"/\u003e\u003c/svg\u003e\r\n                Imprimir / Exportar a PDF\r\n            \u003c/button\u003e\r\n        \u003c/div\u003e\r\n    \u003c/div\u003e\r\n\r\n    \u003c!-- ENCABEZADO CIENTÍFICO / TESIS --\u003e\r\n    \u003cheader class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003cdiv class=\"flex flex-col md:flex-row justify-between items-start md:items-center gap-4 border-b border-slate-100 pb-6 mb-6\"\u003e\r\n            \u003cdiv\u003e\r\n                \u003cspan class=\"text-xs font-bold tracking-wider uppercase text-indigo-600 bg-indigo-50 px-3 py-1 rounded-full border border-indigo-100\"\u003e\r\n                    Trabajo Final de Grado • Tesis de Ingeniería\r\n                \u003c/span\u003e\r\n                \u003ch1 class=\"text-2xl md:text-3xl font-extrabold text-slate-900 mt-2 tracking-tight\"\u003e\r\n                    Evaluación Comparativa de Pooling Morfológico Adaptativo (RGB) en Melanoma — {config.get(\u0027model\u0027,\u0027simple\u0027).upper()} ({config.get(\u0027pool\u0027,\u0027max\u0027).upper()} vs Morph)\r\n                \u003c/h1\u003e\r\n                \u003cp class=\"text-sm text-slate-500 mt-1\"\u003e\r\n                    Análisis de desempeño clínico diagnósticos en redes neuronales convolucionales con submuestreo morfológico vectorial vs. estándar marginal.\r\n                \u003c/p\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"text-right\"\u003e\r\n                \u003cdiv class=\"text-xs text-slate-400 font-semibold uppercase\"\u003eAmbiente de Cómputo\u003c/div\u003e\r\n                \u003cdiv class=\"text-sm font-bold text-slate-700 font-mono mt-0.5\"\u003e{config.get(\u0027device\u0027, \u0027CPU/CUDA\u0027).upper()}\u003c/div\u003e\r\n                \u003cdiv class=\"text-xs text-slate-500 mt-1\"\u003eResolución: {config.get(\u0027resolution\u0027, \u0027112x112\u0027)} px\u003c/div\u003e\r\n            \u003c/div\u003e\r\n        \u003c/div\u003e\r\n\r\n        \u003c!-- METADATOS TÉCNICOS DEL EXPERIMENTO --\u003e\r\n        \u003cdiv class=\"grid grid-cols-2 sm:grid-cols-4 md:grid-cols-8 gap-3 text-center\"\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eDataset\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-slate-800\"\u003e{config.get(\u0027dataset\u0027, \u0027ISIC Dermoscopy\u0027)}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eDescriptor \u0026phi;\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-indigo-700 font-mono\"\u003e{config.get(\u0027metric\u0027, \u0027SMO\u0027)}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003ePartición Dominio\u003c/span\u003e\r\n                \u003cspan class=\"text-xs font-bold text-indigo-700 font-mono\"\u003e{config.get(\u0027domain_label\u0027, f\"B{config.get(\u0027stat_window\u0027, 2)}{config.get(\u0027partition\u0027, \u0027W0\u0027)}\")}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eMuestras Train\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-slate-800 font-mono\"\u003e{config.get(\u0027train_samples\u0027, \u0027N/A\u0027)}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eMuestras Test\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-slate-800 font-mono\"\u003e{config.get(\u0027test_samples\u0027, \u0027N/A\u0027)}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eÉpocas\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-slate-800 font-mono\"\u003e{config.get(\u0027epochs\u0027, 5)}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eBatch Size\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-slate-800 font-mono\"\u003e{config.get(\u0027batch_size\u0027, 64)}\u003c/span\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"bg-slate-50 p-3 rounded-xl border border-slate-100\"\u003e\r\n                \u003cspan class=\"text-xs text-slate-500 block font-medium\"\u003eSemilla (Seed)\u003c/span\u003e\r\n                \u003cspan class=\"text-sm font-bold text-indigo-600 font-mono\"\u003e{config.get(\u0027seed\u0027, 42)} (Fija)\u003c/span\u003e\r\n            \u003c/div\u003e\r\n        \u003c/div\u003e\r\n    \u003c/header\u003e\r\n\r\n    \u003c!-- TARJETAS DE MÉTRICAS CLÍNICAS PRINCIPALES (SCORECARD) --\u003e\r\n    \u003csection class=\"mb-8\"\u003e\r\n        \u003ch2 class=\"text-lg font-bold text-slate-900 mb-4 flex items-center\"\u003e\r\n            \u003csvg class=\"w-5 h-5 mr-2 text-indigo-600\" fill=\"none\" stroke=\"currentColor\" viewBox=\"0 0 24 24\"\u003e\u003cpath stroke-linecap=\"round\" stroke-linejoin=\"round\" stroke-width=\"2\" d=\"M9 19v-6a2 2 0 00-2-2H5a2 2 0 00-2 2v6a2 2 0 002 2h2a2 2 0 002-2zm0 0V9a2 2 0 012-2h2a2 2 0 012 2v10m-6 0a2 2 0 002 2h2a2 2 0 002-2m0 0V5a2 2 0 012-2h2a2 2 0 012 2v14a2 2 0 01-2 2h-2a2 2 0 01-2-2z\"/\u003e\u003c/svg\u003e\r\n            Indicadores Clave de Desempeño Diagnóstico (Test Ciego)\r\n        \u003c/h2\u003e\r\n        \u003cdiv class=\"grid grid-cols-1 sm:grid-cols-2 lg:grid-cols-4 gap-4\"\u003e\r\n            \r\n            \u003c!-- ACCURACY --\u003e\r\n            \u003cdiv class=\"bg-white border border-slate-200 rounded-xl p-5 shadow-sm\"\u003e\r\n                \u003cdiv class=\"flex justify-between items-start\"\u003e\r\n                    \u003cspan class=\"text-xs font-semibold uppercase tracking-wider text-slate-400\"\u003eExactitud (Accuracy)\u003c/span\u003e\r\n                    {badge(d_acc)}\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"text-2xl font-black text-slate-900 mt-2 font-mono\"\u003e{proposed[\u0027acc\u0027]*100:.2f}%\u003c/div\u003e\r\n                \u003cdiv class=\"text-xs text-slate-500 mt-1 flex justify-between\"\u003e\r\n                    \u003cspan\u003eBaseline: \u003cspan class=\"font-mono\"\u003e{baseline[\u0027acc\u0027]*100:.2f}%\u003c/span\u003e\u003c/span\u003e\r\n                    \u003cspan class=\"text-indigo-600 font-medium\"\u003eΔ {(d_acc)*100:+.2f}%\u003c/span\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n\r\n            \u003c!-- SENSIBILIDAD / RECALL --\u003e\r\n            \u003cdiv class=\"bg-white border-2 border-indigo-200 rounded-xl p-5 shadow-sm bg-gradient-to-br from-white to-indigo-50/30\"\u003e\r\n                \u003cdiv class=\"flex justify-between items-start\"\u003e\r\n                    \u003cspan class=\"text-xs font-bold uppercase tracking-wider text-indigo-600\"\u003eSensibilidad (Recall) ★\u003c/span\u003e\r\n                    {badge(d_sens)}\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"text-2xl font-black text-indigo-950 mt-2 font-mono\"\u003e{proposed[\u0027sens\u0027]*100:.2f}%\u003c/div\u003e\r\n                \u003cdiv class=\"text-xs text-slate-500 mt-1 flex justify-between\"\u003e\r\n                    \u003cspan\u003eBaseline: \u003cspan class=\"font-mono\"\u003e{baseline[\u0027sens\u0027]*100:.2f}%\u003c/span\u003e\u003c/span\u003e\r\n                    \u003cspan class=\"text-indigo-600 font-medium font-bold\"\u003eDetección Melanoma\u003c/span\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n\r\n            \u003c!-- ESPECIFICIDAD --\u003e\r\n            \u003cdiv class=\"bg-white border border-slate-200 rounded-xl p-5 shadow-sm\"\u003e\r\n                \u003cdiv class=\"flex justify-between items-start\"\u003e\r\n                    \u003cspan class=\"text-xs font-semibold uppercase tracking-wider text-slate-400\"\u003eEspecificidad\u003c/span\u003e\r\n                    {badge(d_spec)}\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"text-2xl font-black text-slate-900 mt-2 font-mono\"\u003e{proposed[\u0027spec\u0027]*100:.2f}%\u003c/div\u003e\r\n                \u003cdiv class=\"text-xs text-slate-500 mt-1 flex justify-between\"\u003e\r\n                    \u003cspan\u003eBaseline: \u003cspan class=\"font-mono\"\u003e{baseline[\u0027spec\u0027]*100:.2f}%\u003c/span\u003e\u003c/span\u003e\r\n                    \u003cspan class=\"text-indigo-600 font-medium\"\u003eΔ {(d_spec)*100:+.2f}%\u003c/span\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n\r\n            \u003c!-- AUC-ROC --\u003e\r\n            \u003cdiv class=\"bg-white border border-slate-200 rounded-xl p-5 shadow-sm\"\u003e\r\n                \u003cdiv class=\"flex justify-between items-start\"\u003e\r\n                    \u003cspan class=\"text-xs font-semibold uppercase tracking-wider text-slate-400\"\u003eÁrea Bajo Curva (AUC)\u003c/span\u003e\r\n                    {badge(d_auc, is_percentage=False)}\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"text-2xl font-black text-slate-900 mt-2 font-mono\"\u003e{proposed[\u0027auc\u0027]:.4f}\u003c/div\u003e\r\n                \u003cdiv class=\"text-xs text-slate-500 mt-1 flex justify-between\"\u003e\r\n                    \u003cspan\u003eBaseline: \u003cspan class=\"font-mono\"\u003e{baseline[\u0027auc\u0027]:.4f}\u003c/span\u003e\u003c/span\u003e\r\n                    \u003cspan class=\"text-indigo-600 font-medium\"\u003eΔ {(d_auc):+.4f}\u003c/span\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n\r\n        \u003c/div\u003e\r\n    \u003c/section\u003e\r\n\r\n    \u003c!-- GRÁFICOS CIENTÍFICOS: PÉRDIDA Y ROC --\u003e\r\n    \u003csection class=\"grid grid-cols-1 lg:grid-cols-2 gap-6 mb-8\"\u003e\r\n        \r\n        \u003c!-- CURVA DE PÉRDIDA --\u003e\r\n        \u003cdiv class=\"bg-white border border-slate-200 rounded-2xl p-6 shadow-sm\"\u003e\r\n            \u003cdiv class=\"flex justify-between items-center mb-4\"\u003e\r\n                \u003cdiv\u003e\r\n                    \u003ch3 class=\"font-bold text-slate-900 text-sm md:text-base\"\u003eEvolución de la Función de Pérdida (BCE Loss)\u003c/h3\u003e\r\n                    \u003cp class=\"text-xs text-slate-500\"\u003eConvergencia época por época en entrenamiento\u003c/p\u003e\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"flex items-center space-x-3 text-xs\"\u003e\r\n                    \u003cspan class=\"flex items-center\"\u003e\u003cspan class=\"w-3 h-1 bg-red-500 rounded mr-1\"\u003e\u003c/span\u003e Baseline\u003c/span\u003e\r\n                    \u003cspan class=\"flex items-center\"\u003e\u003cspan class=\"w-3 h-1 bg-emerald-500 rounded mr-1\"\u003e\u003c/span\u003e Propuesto\u003c/span\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"flex items-center justify-center p-2 bg-slate-50/50 rounded-xl border border-slate-100\"\u003e\r\n                {loss_svg}\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"mt-3 text-xs text-slate-500 flex justify-between font-mono\"\u003e\r\n                \u003cspan\u003eLoss final Baseline: \u003cb\u003e{baseline[\u0027epoch_losses\u0027][-1]:.4f}\u003c/b\u003e\u003c/span\u003e\r\n                \u003cspan\u003eLoss final Propuesto: \u003cb\u003e{proposed[\u0027epoch_losses\u0027][-1]:.4f}\u003c/b\u003e\u003c/span\u003e\r\n            \u003c/div\u003e\r\n        \u003c/div\u003e\r\n\r\n        \u003c!-- CURVA ROC --\u003e\r\n        \u003cdiv class=\"bg-white border border-slate-200 rounded-2xl p-6 shadow-sm\"\u003e\r\n            \u003cdiv class=\"flex justify-between items-center mb-4\"\u003e\r\n                \u003cdiv\u003e\r\n                    \u003ch3 class=\"font-bold text-slate-900 text-sm md:text-base\"\u003eCurva Característica Operativa del Receptor (ROC)\u003c/h3\u003e\r\n                    \u003cp class=\"text-xs text-slate-500\"\u003eCapacidad discriminativa global en conjunto de prueba\u003c/p\u003e\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"flex items-center space-x-3 text-xs\"\u003e\r\n                    \u003cspan class=\"flex items-center\"\u003e\u003cspan class=\"w-3 h-1 bg-red-500 rounded mr-1\"\u003e\u003c/span\u003e {baseline[\u0027auc\u0027]:.3f}\u003c/span\u003e\r\n                    \u003cspan class=\"flex items-center\"\u003e\u003cspan class=\"w-3 h-1 bg-emerald-500 rounded mr-1\"\u003e\u003c/span\u003e {proposed[\u0027auc\u0027]:.3f}\u003c/span\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"flex items-center justify-center p-2 bg-slate-50/50 rounded-xl border border-slate-100\"\u003e\r\n                {roc_svg}\r\n            \u003c/div\u003e\r\n            \u003cdiv class=\"mt-3 text-xs text-slate-500 text-center\"\u003e\r\n                La curva más cercana a la esquina superior izquierda representa el clasificador superior.\r\n            \u003c/div\u003e\r\n        \u003c/div\u003e\r\n\r\n    \u003c/section\u003e\r\n\r\n    \u003c!-- MATRICES DE CONFUSIÓN CLÍNICAS --\u003e\r\n    \u003csection class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003ch2 class=\"text-lg font-bold text-slate-900 mb-2 flex items-center\"\u003e\r\n            \u003csvg class=\"w-5 h-5 mr-2 text-indigo-600\" fill=\"none\" stroke=\"currentColor\" viewBox=\"0 0 24 24\"\u003e\u003cpath stroke-linecap=\"round\" stroke-linejoin=\"round\" stroke-width=\"2\" d=\"M4 6a2 2 0 012-2h2a2 2 0 012 2v2a2 2 0 01-2 2H6a2 2 0 01-2-2V6zM14 6a2 2 0 012-2h2a2 2 0 012 2v2a2 2 0 01-2 2h-2a2 2 0 01-2-2V6zM4 16a2 2 0 012-2h2a2 2 0 012 2v2a2 2 0 01-2 2H6a2 2 0 01-2-2v-2zM14 16a2 2 0 012-2h2a2 2 0 012 2v2a2 2 0 01-2 2h-2a2 2 0 01-2-2v-2z\"/\u003e\u003c/svg\u003e\r\n            Análisis de Matrices de Confusión y Repercusión Clínica\r\n        \u003c/h2\u003e\r\n        \u003cp class=\"text-xs text-slate-500 mb-6\"\u003e\r\n            En oncología dermatológica, minimizar los \u003cb\u003eFalsos Negativos (Melanomas pasados por alto)\u003c/b\u003e es la prioridad clínica más alta para evitar fatalidades.\r\n        \u003c/p\u003e\r\n\r\n        \u003cdiv class=\"grid grid-cols-1 md:grid-cols-2 gap-8\"\u003e\r\n            \r\n            \u003c!-- MATRIZ BASELINE --\u003e\r\n            \u003cdiv class=\"border border-slate-200 rounded-xl p-5 bg-slate-50/60\"\u003e\r\n                \u003cdiv class=\"flex justify-between items-center mb-3\"\u003e\r\n                    \u003ch3 class=\"font-bold text-slate-800 text-sm\"\u003e{baseline[\u0027name\u0027]}\u003c/h3\u003e\r\n                    \u003cspan class=\"text-xs text-slate-500\"\u003eTiempo: {baseline[\u0027elapsed\u0027]:.1f}s\u003c/span\u003e\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"grid grid-cols-2 gap-3 text-center\"\u003e\r\n                    \u003cdiv class=\"cm-cell bg-emerald-50 border border-emerald-200 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-emerald-700 font-semibold block\"\u003eVerdadero Negativo (Sano)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-emerald-900 font-mono mt-1 block\"\u003e{baseline[\u0027tn\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-emerald-600\"\u003e{(baseline[\u0027tn\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                    \u003cdiv class=\"cm-cell bg-amber-50 border border-amber-200 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-amber-700 font-semibold block\"\u003eFalso Positivo (Biopsia evitable)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-amber-900 font-mono mt-1 block\"\u003e{baseline[\u0027fp\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-amber-600\"\u003e{(baseline[\u0027fp\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                    \u003cdiv class=\"cm-cell bg-rose-50 border-2 border-rose-300 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-rose-700 font-bold block\"\u003eFalso Negativo (¡Peligro!)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-rose-900 font-mono mt-1 block\"\u003e{baseline[\u0027fn\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-rose-600\"\u003e{(baseline[\u0027fn\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                    \u003cdiv class=\"cm-cell bg-emerald-50 border border-emerald-200 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-emerald-700 font-semibold block\"\u003eVerdadero Positivo (Melanoma)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-emerald-900 font-mono mt-1 block\"\u003e{baseline[\u0027tp\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-emerald-600\"\u003e{(baseline[\u0027tp\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n\r\n            \u003c!-- MATRIZ PROPUESTO --\u003e\r\n            \u003cdiv class=\"border-2 border-indigo-200 rounded-xl p-5 bg-indigo-50/30\"\u003e\r\n                \u003cdiv class=\"flex justify-between items-center mb-3\"\u003e\r\n                    \u003ch3 class=\"font-bold text-indigo-950 text-sm flex items-center\"\u003e\r\n                        \u003cspan\u003e{proposed[\u0027name\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"ml-2 text-[10px] bg-indigo-100 text-indigo-700 px-2 py-0.5 rounded font-bold uppercase\"\u003ePropuesta\u003c/span\u003e\r\n                    \u003c/h3\u003e\r\n                    \u003cspan class=\"text-xs text-slate-500\"\u003eTiempo: {proposed[\u0027elapsed\u0027]:.1f}s\u003c/span\u003e\r\n                \u003c/div\u003e\r\n                \u003cdiv class=\"grid grid-cols-2 gap-3 text-center\"\u003e\r\n                    \u003cdiv class=\"cm-cell bg-emerald-50 border border-emerald-200 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-emerald-700 font-semibold block\"\u003eVerdadero Negativo (Sano)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-emerald-900 font-mono mt-1 block\"\u003e{proposed[\u0027tn\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-emerald-600\"\u003e{(proposed[\u0027tn\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                    \u003cdiv class=\"cm-cell bg-amber-50 border border-amber-200 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-amber-700 font-semibold block\"\u003eFalso Positivo (Biopsia evitable)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-amber-900 font-mono mt-1 block\"\u003e{proposed[\u0027fp\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-amber-600\"\u003e{(proposed[\u0027fp\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                    \u003cdiv class=\"cm-cell bg-rose-50 border-2 border-rose-400 p-4 rounded-lg shadow-sm\"\u003e\r\n                        \u003cspan class=\"text-xs text-rose-700 font-bold block\"\u003eFalso Negativo (¡Peligro!)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-rose-900 font-mono mt-1 block\"\u003e{proposed[\u0027fn\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-rose-600\"\u003e\r\n                            {f\"Reducción: {baseline[\u0027fn\u0027] - proposed[\u0027fn\u0027]} casos\" if proposed[\u0027fn\u0027] \u003c baseline[\u0027fn\u0027] else f\"Diferencia: {proposed[\u0027fn\u0027] - baseline[\u0027fn\u0027]}\"}\r\n                        \u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                    \u003cdiv class=\"cm-cell bg-emerald-50 border border-emerald-200 p-4 rounded-lg\"\u003e\r\n                        \u003cspan class=\"text-xs text-emerald-700 font-semibold block\"\u003eVerdadero Positivo (Melanoma)\u003c/span\u003e\r\n                        \u003cspan class=\"text-xl font-black text-emerald-900 font-mono mt-1 block\"\u003e{proposed[\u0027tp\u0027]}\u003c/span\u003e\r\n                        \u003cspan class=\"text-[11px] text-emerald-600\"\u003e{(proposed[\u0027tp\u0027]/total_test)*100:.1f}%\u003c/span\u003e\r\n                    \u003c/div\u003e\r\n                \u003c/div\u003e\r\n            \u003c/div\u003e\r\n\r\n        \u003c/div\u003e\r\n    \u003c/section\u003e\r\n\r\n    \u003c!-- TABLA CIENTÍFICA COMPARATIVA FORMAL --\u003e\r\n    \u003csection class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003ch2 class=\"text-lg font-bold text-slate-900 mb-4 flex items-center\"\u003e\r\n            \u003csvg class=\"w-5 h-5 mr-2 text-indigo-600\" fill=\"none\" stroke=\"currentColor\" viewBox=\"0 0 24 24\"\u003e\u003cpath stroke-linecap=\"round\" stroke-linejoin=\"round\" stroke-width=\"2\" d=\"M3 10h18M3 14h18m-9-4v8m-7 0h14a2 2 0 002-2V6a2 2 0 00-2-2H5a2 2 0 00-2 2v8a2 2 0 002 2z\"/\u003e\u003c/svg\u003e\r\n            Tabla Resumen de Resultados Experimentales\r\n        \u003c/h2\u003e\r\n        \u003cdiv class=\"overflow-x-auto\"\u003e\r\n            \u003ctable class=\"w-full text-left border-collapse text-sm\"\u003e\r\n                \u003cthead\u003e\r\n                    \u003ctr class=\"border-b border-slate-200 bg-slate-50 text-slate-600 font-semibold text-xs uppercase tracking-wider\"\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eMétrica Experimental\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eRed A: {baseline[\u0027name\u0027]}\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4 text-indigo-900 bg-indigo-50/50\"\u003eRed B: {proposed[\u0027name\u0027]}\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eDiferencia Absoluta (Δ)\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eImpacto Relativo\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eVeredicto Clínico\u003c/th\u003e\r\n                    \u003c/tr\u003e\r\n                \u003c/thead\u003e\r\n                \u003ctbody class=\"divide-y divide-slate-100 font-mono text-xs\"\u003e\r\n                    \u003ctr\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans font-medium text-slate-900\"\u003eExactitud (Accuracy)\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 text-slate-600\"\u003e{baseline[\u0027acc\u0027]*100:.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-bold text-indigo-700 bg-indigo-50/20\"\u003e{proposed[\u0027acc\u0027]*100:.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{d_acc*100:+.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{badge(d_acc)}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans text-slate-700\"\u003e{\u0027Superior\u0027 if d_acc \u003e 0 else \u0027Inferior\u0027}\u003c/td\u003e\r\n                    \u003c/tr\u003e\r\n                    \u003ctr\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans font-medium text-slate-900\"\u003eSensibilidad (Recall)\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 text-slate-600\"\u003e{baseline[\u0027sens\u0027]*100:.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-bold text-indigo-700 bg-indigo-50/20\"\u003e{proposed[\u0027sens\u0027]*100:.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{d_sens*100:+.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{badge(d_sens)}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans text-slate-700\"\u003e{\u0027Mayor detección maligna\u0027 if d_sens \u003e= 0 else \u0027Menor detección\u0027}\u003c/td\u003e\r\n                    \u003c/tr\u003e\r\n                    \u003ctr\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans font-medium text-slate-900\"\u003eEspecificidad\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 text-slate-600\"\u003e{baseline[\u0027spec\u0027]*100:.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-bold text-indigo-700 bg-indigo-50/20\"\u003e{proposed[\u0027spec\u0027]*100:.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{d_spec*100:+.2f}%\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{badge(d_spec)}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans text-slate-700\"\u003e{\u0027Menos falsas alarmas\u0027 if d_spec \u003e= 0 else \u0027Más falsas alarmas\u0027}\u003c/td\u003e\r\n                    \u003c/tr\u003e\r\n                    \u003ctr\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans font-medium text-slate-900\"\u003eAUC-ROC Score\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 text-slate-600\"\u003e{baseline[\u0027auc\u0027]:.4f}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-bold text-indigo-700 bg-indigo-50/20\"\u003e{proposed[\u0027auc\u0027]:.4f}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{d_auc:+.4f}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{badge(d_auc, is_percentage=False)}\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans text-slate-700\"\u003e{\u0027Mayor separabilidad\u0027 if d_auc \u003e= 0 else \u0027Menor separabilidad\u0027}\u003c/td\u003e\r\n                    \u003c/tr\u003e\r\n                    \u003ctr\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans font-medium text-slate-900\"\u003eTiempo de Entrenamiento\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 text-slate-600\"\u003e{baseline[\u0027elapsed\u0027]:.2f}s\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-bold text-indigo-700 bg-indigo-50/20\"\u003e{proposed[\u0027elapsed\u0027]:.2f}s\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e{time_diff:+.2f}s\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4\"\u003e\u003cspan class=\"text-slate-500 font-sans text-xs\"\u003e{(proposed[\u0027elapsed\u0027]/baseline[\u0027elapsed\u0027]):.2f}x\u003c/span\u003e\u003c/td\u003e\r\n                        \u003ctd class=\"py-3 px-4 font-sans text-slate-700\"\u003eCómputo en Python/Unfold\u003c/td\u003e\r\n                    \u003c/tr\u003e\r\n                \u003c/tbody\u003e\r\n            \u003c/table\u003e\r\n        \u003c/div\u003e\r\n    \u003c/section\u003e\r\n\r\n    \u003c!-- FUNDAMENTACIÓN TEÓRICA Y DISCUSIÓN PARA LA DEFENSA DE LA TESIS --\u003e\r\n    \u003csection class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003ch2 class=\"text-lg font-bold text-slate-900 mb-3 flex items-center\"\u003e\r\n            \u003csvg class=\"w-5 h-5 mr-2 text-indigo-600\" fill=\"none\" stroke=\"currentColor\" viewBox=\"0 0 24 24\"\u003e\u003cpath stroke-linecap=\"round\" stroke-linejoin=\"round\" stroke-width=\"2\" d=\"M12 6.253v13m0-13C10.832 5.477 9.246 5 7.5 5S4.168 5.477 3 6.253v13C4.168 18.477 5.754 18 7.5 18s3.332.477 4.5 1.253m0-13C13.168 5.477 14.754 5 16.5 5c1.747 0 3.332.477 4.5 1.253v13C19.832 18.477 18.247 18 16.5 18c-1.746 0-3.332.477-4.5 1.253\"/\u003e\u003c/svg\u003e\r\n            Discusión Teórica y Conclusiones para la Tesis\r\n        \u003c/h2\u003e\r\n        \u003cdiv class=\"prose prose-slate max-w-none text-xs md:text-sm text-slate-600 space-y-4\"\u003e\r\n            \u003cp\u003e\r\n                \u003cb\u003e1. Mecanismo de Submuestreo y Coherencia Cromática:\u003c/b\u003e El operador estándar \u003ccode\u003eMaxPool2d\u003c/code\u003e en imágenes tridimensionales de color RGB realiza un ordenamiento marginal independiente en cada canal (R, G, B). Esto provoca la aparición de \u003ci\u003ecolores espurios o falsos\u003c/i\u003e que no existían en la lesión original, distorsionando características dermoscópicas críticas como la red pigmentaria atípica, glóbulos y el velo azul-blanco. Por el contrario, la capa propuesta \u003ccode\u003eAdaptiveMorphPooling2d\u003c/code\u003e implementa un \u003cb\u003eordenamiento vectorial reducido (h-ordering)\u003c/b\u003e mediante el vector de ponderación estadística \u0026psi; basado en la medida de suavidad/dispersión (SMO), garantizando la selección de un \u003cb\u003evector de color real\u003c/b\u003e presente en la ventana.\r\n            \u003c/p\u003e\r\n            \u003cp\u003e\r\n                \u003cb\u003e2. Rigor Experimental (A/B Testing Ceteris Paribus):\u003c/b\u003e Al fijar la semilla aleatoria en $42$ al inicio de cada evaluación, se asegura la paridad estricta en la inicialización de los pesos de las capas convolucionales (\u003ccode\u003efeatures\u003c/code\u003e) y densas (\u003ccode\u003eclassifier\u003c/code\u003e), así como el orden idéntico de entrega de lotes de entrenamiento. De este modo, la variación en el rendimiento se aísla exclusivamente a la función de submuestreo empleada.\r\n            \u003c/p\u003e\r\n            \u003cp\u003e\r\n                \u003cb\u003e3. Compromiso Rendimiento vs. Coste Computacional:\u003c/b\u003e Si bien la formulación actual en PyTorch mediante \u003ccode\u003eF.unfold\u003c/code\u003e introduce una pequeña sobrecarga temporal frente al kernel C++/CUDA nativo y altamente optimizado de \u003ccode\u003eMaxPool2d\u003c/code\u003e, la ganancia observada en métricas diagnósticas justifica la inclusión del pooling morfológico adaptativo en etapas tempranas de redes orientadas al análisis de patologías dermatológicas.\r\n            \u003c/p\u003e\r\n            \u003cp\u003e\r\n                \u003cb\u003e4. Consideración Metodológica sobre Normalización entre Arquitecturas:\u003c/b\u003e En la CNN simple los tensores de entrada se mantienen en el rango $[0, 1]$ (\u003ccode\u003eToTensor()\u003c/code\u003e), mientras que en los modelos preentrenados (VGG-16 y ResNet-18) se aplica la normalización estándar de ImageNet (centrada en cero con valores negativos). Debido a esto, para comparar arquitecturas de forma metodológicamente consistente se deben emplear los descriptores basados en dispersión (\u003ccode\u003eSMO\u003c/code\u003e o \u003ccode\u003eVAR\u003c/code\u003e, invariantes a desplazamientos de media), evitando descriptores sensibles al rango y signo como \u003ccode\u003eMIN\u003c/code\u003e, \u003ccode\u003eMAX\u003c/code\u003e o \u003ccode\u003eMEAN\u003c/code\u003e.\r\n            \u003c/p\u003e\r\n        \u003c/div\u003e\r\n    \u003c/section\u003e\r\n\r\n    \u003c!-- PIE DE PÁGINA --\u003e\r\n    \u003cfooter class=\"text-center text-xs text-slate-400 py-6 border-t border-slate-200\"\u003e\r\n        \u003cp\u003eInforme generado automáticamente por el Entorno Experimental de Tesis • {config.get(\u0027date\u0027, time.strftime(\u0027%Y-%m-%d\u0027))}\u003c/p\u003e\r\n        \u003cp class=\"mt-1\"\u003eImplementación en PyTorch con Pooling Morfológico Adaptativo (RGB SMO)\u003c/p\u003e\r\n    \u003c/footer\u003e\r\n\r\n\u003c/body\u003e\r\n\u003c/html\u003e\r\n\u0027\u0027\u0027\r\n    reports_dir = \"reportes\"\r\n    if not os.path.dirname(output_path):\r\n        output_path = os.path.join(reports_dir, output_path)\r\n    os.makedirs(os.path.dirname(output_path), exist_ok=True)\r\n\r\n    with open(output_path, \"w\", encoding=\"utf-8\") as f:\r\n        f.write(html_content)\r\n\r\n    return os.path.abspath(output_path)\r\n","report_aggregator.py":"import os\r\nimport json\r\nimport time\r\nimport math\r\nimport statistics as st\r\nimport numpy as np\r\n\r\ntry:\r\n    from scipy.stats import ttest_rel as _scipy_ttest_rel\r\nexcept ImportError:\r\n    _scipy_ttest_rel = None\r\n\r\n\r\ndef _paired_ttest(prop_vals, base_vals):\r\n    \"\"\"\r\n    Calcula el t-test pareado (prop_vals vs base_vals).\r\n    Usa scipy.stats.ttest_rel si está disponible; de lo contrario, aplica el cálculo exacto.\r\n    \"\"\"\r\n    n = len(prop_vals)\r\n    if n \u003c 2:\r\n        return 0.0, 1.0\r\n\r\n    diffs = np.asarray(prop_vals, dtype=np.float64) - np.asarray(base_vals, dtype=np.float64)\r\n    if np.allclose(diffs, diffs[0]):\r\n        if abs(diffs[0]) \u003c 1e-12:\r\n            return 0.0, 1.0\r\n        return float(\"inf\") if diffs[0] \u003e 0 else float(\"-inf\"), 0.0\r\n\r\n    if _scipy_ttest_rel is not None:\r\n        res = _scipy_ttest_rel(prop_vals, base_vals)\r\n        t_stat = float(res.statistic) if not np.isnan(res.statistic) else 0.0\r\n        p_val = float(res.pvalue) if not np.isnan(res.pvalue) else 1.0\r\n        return t_stat, p_val\r\n\r\n    # Fallback analítico si scipy no está instalado\r\n    mean_d = float(np.mean(diffs))\r\n    std_d = float(np.std(diffs, ddof=1))\r\n    if std_d \u003c 1e-15:\r\n        return 0.0, 1.0\r\n    t_stat = mean_d / (std_d / math.sqrt(n))\r\n    # Aproximación normal bilateral para p-value en caso de no contar con scipy\r\n    p_val = math.erfc(abs(t_stat) / math.sqrt(2.0))\r\n    return t_stat, p_val\r\n\r\n\r\ndef aggregate_report(results, output_path=\"reporte_agregado_semillas.html\", config=None):\r\n    \"\"\"\r\n    Genera un informe HTML científico agregado a partir de múltiples semillas (seeds),\r\n    calculando media, desviación estándar y p-value mediante t-test pareado.\r\n\r\n    Parámetros:\r\n        results: lista de diccionarios [{\"seed\": 42, \"epochs\": 10, \"baseline\": {...}, \"proposed\": {...}}, ...]\r\n        output_path: nombre o ruta del archivo HTML de salida (se guarda en \u0027reportes/\u0027)\r\n        config: diccionario opcional con metadatos del experimento\r\n    \"\"\"\r\n    config = config or {}\r\n    reports_dir = \"reportes\"\r\n    if not os.path.dirname(output_path):\r\n        output_path = os.path.join(reports_dir, output_path)\r\n    os.makedirs(os.path.dirname(output_path), exist_ok=True)\r\n\r\n    # Guardar también respaldo en JSON\r\n    json_path = os.path.splitext(output_path)[0] + \".json\"\r\n    with open(json_path, \"w\", encoding=\"utf-8\") as jf:\r\n        json.dump({\"config\": config, \"results\": results}, jf, indent=2)\r\n\r\n    n_seeds = len(results)\r\n    seeds_str = \", \".join(str(r.get(\"seed\", \"?\")) for r in results)\r\n\r\n    metrics_spec = [\r\n        (\"AUC-ROC\", \"auc\", False),\r\n        (\"Accuracy (Exactitud)\", \"acc\", True),\r\n        (\"Sensibilidad (Recall)\", \"sens\", True),\r\n        (\"Especificidad\", \"spec\", True),\r\n    ]\r\n\r\n    summary_rows = []\r\n    for label, key, is_pct in metrics_spec:\r\n        scale = 100.0 if is_pct else 1.0\r\n        unit = \"%\" if is_pct else \"\"\r\n        fmt = \".2f\" if is_pct else \".4f\"\r\n\r\n        base_vals = [r[\"baseline\"][key] * scale for r in results]\r\n        prop_vals = [r[\"proposed\"][key] * scale for r in results]\r\n        deltas = [p - b for p, b in zip(prop_vals, base_vals)]\r\n\r\n        mean_base = st.mean(base_vals)\r\n        std_base = st.stdev(base_vals) if n_seeds \u003e 1 else 0.0\r\n        mean_prop = st.mean(prop_vals)\r\n        std_prop = st.stdev(prop_vals) if n_seeds \u003e 1 else 0.0\r\n        mean_delta = st.mean(deltas)\r\n        std_delta = st.stdev(deltas) if n_seeds \u003e 1 else 0.0\r\n\r\n        t_stat, p_value = _paired_ttest(prop_vals, base_vals)\r\n\r\n        if n_seeds \u003c 2:\r\n            verdict = \u0027\u003cspan class=\"px-2.5 py-1 rounded text-xs font-semibold bg-slate-100 text-slate-600\"\u003eSe requieren ≥2 seeds\u003c/span\u003e\u0027\r\n        elif p_value \u003c 0.05 and mean_delta \u003e 0:\r\n            verdict = \u0027\u003cspan class=\"px-2.5 py-1 rounded text-xs font-semibold bg-emerald-100 text-emerald-800 border border-emerald-300\"\u003e✅ Significativo (Ganancia)\u003c/span\u003e\u0027\r\n        elif p_value \u003c 0.05 and mean_delta \u003c 0:\r\n            verdict = \u0027\u003cspan class=\"px-2.5 py-1 rounded text-xs font-semibold bg-rose-100 text-rose-800 border border-rose-300\"\u003e❌ Significativo (Inferior)\u003c/span\u003e\u0027\r\n        else:\r\n            verdict = \u0027\u003cspan class=\"px-2.5 py-1 rounded text-xs font-semibold bg-amber-100 text-amber-800 border border-amber-300\"\u003e⚠️ No significativo\u003c/span\u003e\u0027\r\n\r\n        delta_cls = \"text-emerald-700 font-bold\" if mean_delta \u003e= 0 else \"text-rose-700 font-bold\"\r\n        p_cls = \"text-emerald-700 font-bold\" if (n_seeds \u003e 1 and p_value \u003c 0.05) else \"text-slate-600\"\r\n\r\n        summary_rows.append(f\"\"\"\r\n        \u003ctr class=\"hover:bg-slate-50/80\"\u003e\r\n            \u003ctd class=\"py-3.5 px-4 font-sans font-semibold text-slate-900\"\u003e{label}\u003c/td\u003e\r\n            \u003ctd class=\"py-3.5 px-4 text-slate-600\"\u003e{mean_base:{fmt}}{unit} ± {std_base:{fmt}}{unit}\u003c/td\u003e\r\n            \u003ctd class=\"py-3.5 px-4 font-semibold text-indigo-700 bg-indigo-50/30\"\u003e{mean_prop:{fmt}}{unit} ± {std_prop:{fmt}}{unit}\u003c/td\u003e\r\n            \u003ctd class=\"py-3.5 px-4 {delta_cls}\"\u003e{mean_delta:+{fmt}}{unit}\u003c/td\u003e\r\n            \u003ctd class=\"py-3.5 px-4 text-slate-600\"\u003e±{std_delta:{fmt}}{unit}\u003c/td\u003e\r\n            \u003ctd class=\"py-3.5 px-4 {p_cls}\"\u003e{p_value:.4f}\u003c/td\u003e\r\n            \u003ctd class=\"py-3.5 px-4 font-sans\"\u003e{verdict}\u003c/td\u003e\r\n        \u003c/tr\u003e\r\n        \"\"\")\r\n\r\n    seed_rows = []\r\n    for r in results:\r\n        s = r.get(\"seed\", \"N/A\")\r\n        b = r[\"baseline\"]\r\n        p = r[\"proposed\"]\r\n        d_auc = p[\"auc\"] - b[\"auc\"]\r\n        d_acc = (p[\"acc\"] - b[\"acc\"]) * 100.0\r\n        d_sens = (p[\"sens\"] - b[\"sens\"]) * 100.0\r\n        d_spec = (p[\"spec\"] - b[\"spec\"]) * 100.0\r\n        seed_rows.append(f\"\"\"\r\n        \u003ctr class=\"hover:bg-slate-50/80\"\u003e\r\n            \u003ctd class=\"py-2.5 px-4 font-bold text-indigo-600\"\u003e{s}\u003c/td\u003e\r\n            \u003ctd class=\"py-2.5 px-4\"\u003e{b[\u0027auc\u0027]:.4f} → \u003cb\u003e{p[\u0027auc\u0027]:.4f}\u003c/b\u003e ({d_auc:+.4f})\u003c/td\u003e\r\n            \u003ctd class=\"py-2.5 px-4\"\u003e{b[\u0027acc\u0027]*100:.2f}% → \u003cb\u003e{p[\u0027acc\u0027]*100:.2f}%\u003c/b\u003e ({d_acc:+.2f}%)\u003c/td\u003e\r\n            \u003ctd class=\"py-2.5 px-4\"\u003e{b[\u0027sens\u0027]*100:.2f}% → \u003cb\u003e{p[\u0027sens\u0027]*100:.2f}%\u003c/b\u003e ({d_sens:+.2f}%)\u003c/td\u003e\r\n            \u003ctd class=\"py-2.5 px-4\"\u003e{b[\u0027spec\u0027]*100:.2f}% → \u003cb\u003e{p[\u0027spec\u0027]*100:.2f}%\u003c/b\u003e ({d_spec:+.2f}%)\u003c/td\u003e\r\n        \u003c/tr\u003e\r\n        \"\"\")\r\n\r\n    model_title = config.get(\"model\", \"simple\").upper()\r\n    pool_title = f\"{config.get(\u0027pool\u0027, \u0027max\u0027).upper()} vs Morph\"\r\n    metric_name = config.get(\"metric\", \"SMO\")\r\n    stat_win = config.get(\"stat_window\", \"2\")\r\n    partition_name = config.get(\"partition\", \"W0\")\r\n    domain_desc = config.get(\"domain_label\", f\"Ventana {stat_win} · {partition_name}\")\r\n    epochs_val = config.get(\"epochs\", \"N/A\")\r\n    dataset_name = config.get(\"dataset\", \"ISIC Dermoscopy\")\r\n    df_val = max(0, n_seeds - 1)\r\n    engine_label = \"scipy.stats.ttest_rel (Distribución t de Student exacta)\" if _scipy_ttest_rel is not None else \"Fallback analítico (ADVERTENCIA: instalar scipy para distribución t exacta)\"\r\n\r\n    power_warning_html = \"\"\r\n    if n_seeds \u003c 10:\r\n        power_warning_html = f\"\"\"\r\n    \u003csection class=\"bg-amber-50 border border-amber-200 rounded-2xl p-5 md:p-6 mb-8 shadow-sm\"\u003e\r\n        \u003ch3 class=\"text-sm font-bold text-amber-900 mb-1.5\"\u003e\r\n            ⚠️ Nota Metodológica sobre Potencia Estadística (n = {n_seeds} \u0026lt; 10, gl = {df_val})\r\n        \u003c/h3\u003e\r\n        \u003cp class=\"text-xs md:text-sm text-amber-800 leading-relaxed\"\u003e\r\n            El test \u003ci\u003et\u003c/i\u003e de Student pareado se calculó con \u003cb\u003en = {n_seeds} semillas\u003c/b\u003e (grados de libertad \u003cb\u003egl = {df_val}\u003c/b\u003e) empleando \u003ccode\u003e{engine_label}\u003c/code\u003e.\r\n            Con tamaños muestrales pequeños (\u003cb\u003en \u0026lt; 10\u003c/b\u003e), la distribución \u003ci\u003et\u003c/i\u003e posee colas pesadas y una \u003cb\u003epotencia estadística reducida (1 − β)\u003c/b\u003e, lo que incrementa la probabilidad de Error Tipo II (no detectar significancia estadística cuando la mejora es moderada).\r\n            En consecuencia: (1) un resultado \u003cb\u003ep \u0026lt; 0.05\u003c/b\u003e con pocas semillas constituye evidencia sólida de un tamaño de efecto altamente consistente, mientras que (2) un resultado \u003cb\u003ep ≥ 0.05\u003c/b\u003e con Δ medio positivo sugiere tendencia favorable que requiere ampliar a \u003cb\u003en ≥ 10 semillas\u003c/b\u003e para confirmar significancia.\r\n        \u003c/p\u003e\r\n    \u003c/section\u003e\r\n\"\"\"\r\n\r\n    html = f\"\"\"\u003c!DOCTYPE html\u003e\r\n\u003chtml lang=\"es\"\u003e\r\n\u003chead\u003e\r\n    \u003cmeta charset=\"UTF-8\"\u003e\r\n    \u003cmeta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\"\u003e\r\n    \u003ctitle\u003eInforme Estadístico Agregado (Multi-Seed) — {model_title} ({pool_title})\u003c/title\u003e\r\n    \u003cscript src=\"https://cdn.tailwindcss.com\"\u003e\u003c/script\u003e\r\n    \u003clink href=\"https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800\u0026family=JetBrains+Mono:wght@400;600\u0026display=swap\" rel=\"stylesheet\"\u003e\r\n    \u003cstyle\u003e\r\n        body {{ font-family: \u0027Inter\u0027, sans-serif; background-color: #f8fafc; color: #1e293b; }}\r\n        .font-mono {{ font-family: \u0027JetBrains Mono\u0027, monospace; }}\r\n        @media print {{\r\n            .no-print {{ display: none !important; }}\r\n            body {{ background-color: #ffffff; }}\r\n        }}\r\n    \u003c/style\u003e\r\n\u003c/head\u003e\r\n\u003cbody class=\"p-4 md:p-8 max-w-6xl mx-auto\"\u003e\r\n\r\n    \u003cdiv class=\"no-print flex justify-between items-center mb-6 pb-4 border-b border-slate-200\"\u003e\r\n        \u003cdiv class=\"flex items-center space-x-2 text-sm text-slate-500\"\u003e\r\n            \u003cspan class=\"inline-block w-2.5 h-2.5 rounded-full bg-indigo-600\"\u003e\u003c/span\u003e\r\n            \u003cspan\u003eAnálisis Estadístico Multi-Semilla ({n_seeds} ejecuciones pareadas)\u003c/span\u003e\r\n            \u003cspan\u003e•\u003c/span\u003e\r\n            \u003cspan class=\"font-mono text-xs\"\u003e{time.strftime(\u0027%Y-%m-%d %H:%M:%S\u0027)}\u003c/span\u003e\r\n        \u003c/div\u003e\r\n        \u003cbutton onclick=\"window.print()\" class=\"px-4 py-2 bg-slate-900 hover:bg-slate-800 text-white text-xs font-semibold rounded-lg shadow-sm transition\"\u003e\r\n            Imprimir / Exportar a PDF\r\n        \u003c/button\u003e\r\n    \u003c/div\u003e\r\n\r\n    \u003cheader class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003cspan class=\"text-xs font-bold tracking-wider uppercase text-indigo-600 bg-indigo-50 px-3 py-1 rounded-full border border-indigo-100\"\u003e\r\n            Validación Estadística • t-test Pareado (α = 0.05)\r\n        \u003c/span\u003e\r\n        \u003ch1 class=\"text-2xl md:text-3xl font-extrabold text-slate-900 mt-3 tracking-tight\"\u003e\r\n            Tabla Agregada de Evidencia Estadística — {model_title} ({pool_title})\r\n        \u003c/h1\u003e\r\n        \u003cp class=\"text-sm text-slate-500 mt-1\"\u003e\r\n            Descriptor: \u003cb\u003e{metric_name}\u003c/b\u003e • Dominio: \u003cb\u003e{domain_desc}\u003c/b\u003e • Dataset: \u003cb\u003e{dataset_name}\u003c/b\u003e • Épocas: \u003cb\u003e{epochs_val}\u003c/b\u003e • Semillas evaluadas: \u003cspan class=\"font-mono\"\u003e{seeds_str}\u003c/span\u003e\r\n        \u003c/p\u003e\r\n    \u003c/header\u003e\r\n\r\n    \u003csection class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003ch2 class=\"text-lg font-bold text-slate-900 mb-4\"\u003eResumen Estadístico y Significancia (Paired Student\u0027s t-test)\u003c/h2\u003e\r\n        \u003cdiv class=\"overflow-x-auto\"\u003e\r\n            \u003ctable class=\"w-full text-left border-collapse text-sm\"\u003e\r\n                \u003cthead\u003e\r\n                    \u003ctr class=\"border-b border-slate-200 bg-slate-50 text-slate-600 font-semibold text-xs uppercase tracking-wider\"\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eMétrica\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eBaseline (Media ± Std)\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4 text-indigo-900 bg-indigo-50/50\"\u003ePropuesto MorphPool (Media ± Std)\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eΔ Medio\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eΔ Std\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003ep-value\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eVeredicto (α = 0.05)\u003c/th\u003e\r\n                    \u003c/tr\u003e\r\n                \u003c/thead\u003e\r\n                \u003ctbody class=\"divide-y divide-slate-100 font-mono text-xs\"\u003e\r\n                    {\u0027\u0027.join(summary_rows)}\r\n                \u003c/tbody\u003e\r\n            \u003c/table\u003e\r\n        \u003c/div\u003e\r\n    \u003c/section\u003e\r\n\r\n    \u003csection class=\"bg-white border border-slate-200 rounded-2xl p-6 md:p-8 mb-8 shadow-sm\"\u003e\r\n        \u003ch2 class=\"text-lg font-bold text-slate-900 mb-4\"\u003eDesglose Individual por Semilla (Seed)\u003c/h2\u003e\r\n        \u003cdiv class=\"overflow-x-auto\"\u003e\r\n            \u003ctable class=\"w-full text-left border-collapse text-sm\"\u003e\r\n                \u003cthead\u003e\r\n                    \u003ctr class=\"border-b border-slate-200 bg-slate-50 text-slate-600 font-semibold text-xs uppercase tracking-wider\"\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eSeed\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eAUC (Base → Prop)\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eAccuracy (Base → Prop)\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eSensibilidad (Base → Prop)\u003c/th\u003e\r\n                        \u003cth class=\"py-3 px-4\"\u003eEspecificidad (Base → Prop)\u003c/th\u003e\r\n                    \u003c/tr\u003e\r\n                \u003c/thead\u003e\r\n                \u003ctbody class=\"divide-y divide-slate-100 font-mono text-xs\"\u003e\r\n                    {\u0027\u0027.join(seed_rows)}\r\n                \u003c/tbody\u003e\r\n            \u003c/table\u003e\r\n        \u003c/div\u003e\r\n    \u003c/section\u003e\r\n\r\n    {power_warning_html}\r\n\r\n\u003c/body\u003e\r\n\u003c/html\u003e\r\n\"\"\"\r\n    with open(output_path, \"w\", encoding=\"utf-8\") as f:\r\n        f.write(html)\r\n\r\n    return os.path.abspath(output_path)\r\n\r\n"}''')

for filename, code_text in MODULES.items():
    out_file = os.path.join(WORK_DIR, filename)
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(code_text)
    print(f"✅ Módulo listo: {out_file} ({len(code_text):,} bytes)")

if "DATASETS_DRIVE_DIR" in globals() and DATASETS_DRIVE_DIR and os.path.isdir(DATASETS_DRIVE_DIR):
    DRIVE_REPORTS_DIR = os.path.join(DATASETS_DRIVE_DIR, "reportes_colab")
else:
    DRIVE_REPORTS_DIR = "/content/drive/MyDrive/TESIS_Reportes_MorphPool"

os.makedirs(DRIVE_REPORTS_DIR, exist_ok=True)
print(f"\n💾 Carpeta de guardado automático en Google Drive:\n   📂 {DRIVE_REPORTS_DIR}")

In [ ]:
#@title 🎛️ 3. Panel Desplegable de Configuración y Ejecución A/B { display-mode: "form" }
import os
import random
import shutil
import subprocess

# --- PARÁMETROS SELECCIONABLES (LISTAS DESPLEGABLES DE COLAB) ---
dataset = "archive_2k_2" #@param ["archive", "archive_2k", "archive_2k_2", "archive2", "archive3", "archive11k_1", "archive11k_2", "archive_resized_1", "archive_resized_2", "archive_resized_3"] {allow-input: true}
modelo = "simple" #@param ["simple", "resnet18", "vgg16"]
pool_baseline = "max" #@param ["max", "avg"]
descriptor_phi = "ENT" #@param ["ENT", "SMO", "MEAN", "VAR", "MO1", "MO2", "MIN", "MAX"]
particion_dominio = "W9" #@param ["W0", "W1", "W4", "W9", "W16", "W25", "W36", "W49", "W64"]
ventana_B = "2" #@param ["2", "3", "5", "7", "global"]
epocas = "50" #@param ["5", "10", "15", "25", "50", "100"] {allow-input: true}
batch_size = "Automático" #@param ["Automático", "16", "32", "64", "128"]
learning_rate = "Automático" #@param ["Automático", "0.001", "0.0005", "0.0001", "0.00005"] {allow-input: true}
modo_semillas = "Semilla única" #@param ["Semilla única", "Múltiples semillas (t-test)"]
semillas = "42" #@param {type:"string"}
generar_semillas_aleatorias = 0 #@param {type:"integer"}
usar_tensor_cores_fp16 = True #@param {type:"boolean"}
copiar_carpeta_a_ssd_local = True #@param {type:"boolean"}

from concurrent.futures import ThreadPoolExecutor

def _parallel_copy_tree(src_root, dst_root):
    tasks = []
    for r, dirs, files_list in os.walk(src_root):
        rel = os.path.relpath(r, src_root)
        target_dir = dst_root if rel == "." else os.path.join(dst_root, rel)
        os.makedirs(target_dir, exist_ok=True)
        for fn in files_list:
            s_file = os.path.join(r, fn)
            d_file = os.path.join(target_dir, fn)
            if not os.path.exists(d_file):
                tasks.append((s_file, d_file))
    if not tasks:
        return
    total = len(tasks)
    done = 0
    def _cp(pair):
        shutil.copyfile(pair[0], pair[1])
    with ThreadPoolExecutor(max_workers=12) as pool_ex:
        for _ in pool_ex.map(_cp, tasks, chunksize=32):
            done += 1
            if done % 500 == 0 or done == total:
                print(f"\r   Copiados {done}/{total} archivos al SSD local...", end="", flush=True)
    print()

LOCAL_DATASETS_CACHE = "/content/datasets_local"
os.makedirs(LOCAL_DATASETS_CACHE, exist_ok=True)

def _ensure_clean_dataset_root(base_folder):
    """Asegura que base_folder contenga directamente 'train' y 'test' en minúscula."""
    for root_p, dirs, _ in os.walk(base_folder):
        rel_d = root_p[len(base_folder):].count(os.sep)
        lower_map = {d.lower(): d for d in dirs}
        if "train" in lower_map and "test" in lower_map:
            real_train = os.path.join(root_p, lower_map["train"])
            real_test = os.path.join(root_p, lower_map["test"])
            dst_train = os.path.join(base_folder, "train")
            dst_test = os.path.join(base_folder, "test")
            if os.path.abspath(real_train) != os.path.abspath(dst_train) and not os.path.exists(dst_train):
                os.symlink(real_train, dst_train)
            if os.path.abspath(real_test) != os.path.abspath(dst_test) and not os.path.exists(dst_test):
                os.symlink(real_test, dst_test)
            if os.path.isdir(dst_train) and os.path.isdir(dst_test):
                return base_folder
            return root_p
        if rel_d >= 2:
            dirs[:] = []
    return None

def prepare_dataset_path(ds_name):
    ds_clean = ds_name.strip()
    local_target = os.path.join(LOCAL_DATASETS_CACHE, ds_clean.replace(" ", "_"))

    # 1. Si ya está en el SSD local de Colab en esta sesión, reutilizar de inmediato
    if os.path.isdir(local_target):
        valid = _ensure_clean_dataset_root(local_target)
        if valid:
            print(f"⚡ Usando dataset en caché SSD local: {valid}")
            return valid

    if os.path.isabs(ds_clean) and os.path.isdir(ds_clean):
        valid = _ensure_clean_dataset_root(ds_clean)
        if valid:
            return valid

    if not ("DATASETS_DRIVE_DIR" in globals() and DATASETS_DRIVE_DIR and os.path.isdir(DATASETS_DRIVE_DIR)):
        raise FileNotFoundError("Primero ejecuta la Celda 1 para conectar tu carpeta de Google Drive.")

    entries = os.listdir(DATASETS_DRIVE_DIR)
    norm_key = ds_clean.lower().replace(" ", "").replace("_", "")

    # 2. Buscar primero archivo .zip o .rar en la carpeta de Drive (extracción ultrarrápida)
    for fname in entries:
        base, ext = os.path.splitext(fname)
        if ext.lower() in (".zip", ".rar", ".tar", ".gz"):
            if base.lower() == ds_clean.lower() or base.lower().replace(" ", "").replace("_", "") == norm_key:
                arc_path = os.path.join(DATASETS_DRIVE_DIR, fname)
                print(f"🗜️ Extrayendo '{fname}' desde Drive al SSD local de Colab...")
                os.makedirs(local_target, exist_ok=True)
                if ext.lower() == ".zip":
                    subprocess.run(["unzip", "-q", "-o", arc_path, "-d", local_target], check=True)
                elif ext.lower() == ".rar":
                    subprocess.run(["unrar", "x", "-o+", "-inul", arc_path, local_target], check=True)
                else:
                    shutil.unpack_archive(arc_path, local_target)
                valid = _ensure_clean_dataset_root(local_target)
                if valid:
                    print(f"✅ Dataset extraído y listo: {valid}")
                    return valid

    # 3. Buscar carpeta descomprimida en Google Drive
    for fname in entries:
        full_p = os.path.join(DATASETS_DRIVE_DIR, fname)
        if os.path.isdir(full_p):
            if fname.lower() == ds_clean.lower() or fname.lower().replace(" ", "").replace("_", "") == norm_key:
                if copiar_carpeta_a_ssd_local:
                    print(f"📥 Copiando carpeta '{fname}' en paralelo al SSD local de Colab...")
                    _parallel_copy_tree(full_p, local_target)
                    valid = _ensure_clean_dataset_root(local_target)
                    if valid:
                        return valid
                valid = _ensure_clean_dataset_root(full_p)
                if valid:
                    return valid

    raise FileNotFoundError(
        f"No se encontró '{ds_clean}' en {DATASETS_DRIVE_DIR}.\n"
        f"Archivos/carpetas disponibles en tu Drive: {sorted(entries)}"
    )

resolved_data_dir = prepare_dataset_path(dataset)

# Sincronización inteligente ventana_B <-> particion_dominio
part_eff = particion_dominio
win_eff = ventana_B
if win_eff == "global" and part_eff == "W0":
    part_eff = "W1"
elif part_eff != "W1" and win_eff == "global":
    win_eff = "2"

if generar_semillas_aleatorias and generar_semillas_aleatorias > 0:
    seed_list = [str(s) for s in random.sample(range(10, 1000), k=min(generar_semillas_aleatorias, 900))]
else:
    seed_list = [s for s in semillas.replace(",", " ").split() if s.strip()]
    if not seed_list:
        seed_list = ["42"]

ep_list = [e for e in str(epocas).replace(",", " ").split() if e.strip()]
ds_tag = os.path.basename(dataset.strip()).replace(" ", "_")
out_filename = f"{modelo}_{descriptor_phi}_B{win_eff}{part_eff}_{ds_tag}_{'-'.join(ep_list)}ep.html"

cmd = [
    "python3", "-u", "train_eval.py",
    "--model", modelo,
    "--pool", pool_baseline,
    "--metric", descriptor_phi,
    "--partition", part_eff,
    "--stat-window", win_eff,
    "-d", resolved_data_dir,
    "-o", out_filename,
]

if len(ep_list) == 1:
    cmd += ["-e", ep_list[0]]
else:
    cmd += ["--epochs-list"] + ep_list

if batch_size != "Automático":
    cmd += ["-b", batch_size]

if not learning_rate.lower().startswith("autom") and learning_rate.strip():
    cmd += ["--lr", learning_rate.strip()]

if modo_semillas == "Semilla única" and len(seed_list) == 1 and generar_semillas_aleatorias <= 1:
    cmd += ["-s", seed_list[0]]
else:
    cmd += ["--seeds"] + seed_list

if usar_tensor_cores_fp16:
    cmd += ["--amp"]
else:
    cmd += ["--no-amp"]

print("\n🚀 Ejecutando:", " ".join(cmd))
print("=" * 70)

LAST_GENERATED_REPORTS = []
proc = subprocess.Popen(
    cmd,
    cwd=WORK_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end="", flush=True)
    for prefix in ("Archivo HTML Agregado:", "Archivo HTML:"):
        if prefix in line:
            rep_path = line.split(prefix, 1)[1].strip()
            if os.path.isfile(rep_path):
                LAST_GENERATED_REPORTS.append(rep_path)
                shutil.copy2(rep_path, os.path.join(DRIVE_REPORTS_DIR, os.path.basename(rep_path)))
                json_cand = os.path.splitext(rep_path)[0] + ".json"
                if os.path.isfile(json_cand):
                    shutil.copy2(json_cand, os.path.join(DRIVE_REPORTS_DIR, os.path.basename(json_cand)))

rc = proc.wait()
if rc == 0 and LAST_GENERATED_REPORTS:
    print(f"\n💾 Guardados automáticamente {len(LAST_GENERATED_REPORTS)} informe(s) en Google Drive:\n   📂 {DRIVE_REPORTS_DIR}")

In [ ]:
#@title 📊 4. Visualizar Informe HTML Aquí Mismo o Descargarlo a tu PC { display-mode: "form" }
import os
import html as html_lib
from IPython.display import display, HTML
from google.colab import files

descargar_a_pc = False #@param {type:"boolean"}

if "LAST_GENERATED_REPORTS" in globals() and LAST_GENERATED_REPORTS:
    target_report = LAST_GENERATED_REPORTS[-1]
    print(f"📄 Mostrando informe: {os.path.basename(target_report)}")
    if descargar_a_pc:
        for r in LAST_GENERATED_REPORTS:
            files.download(r)
    with open(target_report, "r", encoding="utf-8") as f:
        html_str = f.read()
    escaped = html_lib.escape(html_str)
    display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="850" style="border:1px solid #cbd5e1; border-radius:12px; background:white;"></iframe>'))
else:
    print("⚠️ Aún no se ha generado ningún reporte en esta sesión. Ejecuta la Celda 3 primero.")